# GNSS Timeseries Project

## Documentation overview

## TODO Summary

- format tooltip (pop-ups are slow) #COMPLETE#
- fix vector scaling (and check for correctness)
- fix vector reference bar
- fix vector coloring (up displacement)
- error ellipses
- look into projections (polar, etc.)
- add support for additional organizations/data sets
- add support for manually selecting data folder storage directory
- implement Docker package management
- Fix case sensitivity on everywhere (especially site lookup)


This notebook reads and displays velocity fields and time series from the NGF (EarthScope data center), UNR (University Nevada Reno), and JPL (Jet Propulsion Laboratory).  Most likely additional packages such as ipyleaflet, pandas, numpy, ipywidgets, matplotlib, geopy, json will need to be installed.  Conda installations from conda-forge should be possible for all packages.

The widget interface at the bottom of the notebook has three cells:

<b>Availability:</b> The “Latest NGF”, “Latest UNR”, and “Latest JPL” buttons must be used the first time the Notebook is run to create the data directory and subdirectories for each center that will used to save JSON files with information about all sites at the centers and time series files from each center as they are requested.  Latest updates should be selected to ensure the most up-to-date information is available.  The Availability check can used to see what is known about a 4-char code site name at each of the centers.

<b>Map:</b> The Map interface allows locations of sites, along with other sites within a user-specified radius to be plotted, along with 3-D velocities if available and selected.  Distance table for the 10 nearest sites to a site can be generated.

<b>Timeseries:</b> The Time series interface allows time series to be detrended, with optionally breaks from the EarthScope database removed.  Time series from different sites and centers can be overlayed.  Different backends can be used for plotting time series, with the ipympl backend allowing zooming and saving and the interactive one allowing identification of points.  The default inline backend should work on all systems.  The other backends may not work on all systems. 

All of the cells in the Notebook can be run, and the widgets and additional documentation will be at the bottom of the Notebook.  The code should run in a Jupyter Notebook or VSCode.

## Internal Code


### Importing Packages

In [ ]:
import sys
import ast
from pathlib import Path

# Note: Added to keep shared folders working when this notebook is run from inside Main Project.
PROJECT_ROOT = Path.cwd().resolve()
if (PROJECT_ROOT.parent / "data" / "NGF" / "NGF_data.json").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"

local_ipyleaflet_packages = PROJECT_ROOT / "ipyleaflet_packages"
if local_ipyleaflet_packages.exists():
    sys.path.insert(0, str(local_ipyleaflet_packages))

try:
    import ipyleaflet
except:
    print('ipyleaflet not found in virtual environment. Installing...')
    %pip install ipyleaflet
    import ipyleaflet
    print('ipyleaflet installed in virtual environment')

from ipyleaflet import (
    Map, CircleMarker, Circle, Polyline, Marker, DivIcon,
    TileLayer, WidgetControl, LayerGroup, basemaps
)

import pandas as pd
import numpy as np
import ipywidgets as wg
from ipywidgets import HBox, VBox, HTML, Layout
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import gridspec
import math
try:
    import geopy.distance
except:
    print("geopy not found in virtual environment. Installing...")
    %pip install geopy
    import geopy.distance
    print("geopy installed in virtual environment")
import json
import os
from os.path import exists
from os import makedirs
from datetime import datetime, timedelta, timezone
import urllib.request
import requests
import re
from gnss_core import (
    build_map_render_plan, detrended as core_detrended, filter_sigmas as core_filter_sigmas,
    fit_ts as core_fit_ts, number_list as core_number_list, parse_bulk_selection as core_parse_bulk_selection,
    parse_map_selection, remove_brac as core_remove_brac, vec_add as core_vec_add,
    vec_sub as core_vec_sub, velocity_endpoint as core_velocity_endpoint,
)
from gnss_map_runtime import stage_layer_update
import textwrap
import webbrowser
import ipympl
from IPython import get_ipython

try:
    import earthscope_sdk
except:
    print("earthscope_sdk not found in virtual environment. Installing...")
    %pip install earthscope_sdk
    import earthscope_sdk
    print("earthscope_sdk installed in virtual environment")
esver = earthscope_sdk.__version__

print("\nPackages loaded successfully \nRunning on the following versions:")
print(f"python: {sys.version.split()[0]}")
print(f"ipyleaflet: {ipyleaflet.__version__}")
print(f"ipympl: {ipympl.__version__}")
print(f"earthscope_sdk: {esver}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"ipywidgets: {wg.__version__}")
print(f"matplotlib: {mpl.__version__}")
print(f"requests: {requests.__version__}")

### EarthScope Verification

In [ ]:
## TODO: EarthScope auth now opens the login/data portal when needed
# Define Earthscope access functions and then see if a test download is to be
# executed. Once access has been tested, the NGF_test variable can be set to False
# to avoid repeated testing and login prompts.
NGF_test = False
# Functions for accessing Earthscope das set that depend on the which version of earthscpoe_sdk
# is installled. These functions must be defined even if the NGF access is not being tested.
if (esver[0] == '0') :

    from earthscope_sdk.auth.device_code_flow import DeviceCodeFlowSimple
    from earthscope_sdk.auth.auth_flow import NoTokensError

    def get_es_file(url: str, directory_to_save_file: object = None, token_path: str = './') -> None:
        """Downloads a file from gage-data.earthscope.org using the EarthScope SDK.

        Args:
            url: url of desired file at gage-data.earthscope.org
            directory_to_save_file: path of directory in which to save the file
            token_path: path of directory in which to save the token

        Raises:
            NoTokensError: if no token is found and the device code flow fails
        """

        if directory_to_save_file is None:
            directory_to_save_file = DATA_DIR / "NGF"

        device_flow = DeviceCodeFlowSimple(Path(token_path))

        try:
            device_flow.get_access_token_refresh_if_necessary()
        except NoTokensError:
            device_flow.do_flow()
        token = device_flow.access_token

        file_name = Path(url).name
        r = requests.get(url, headers={"authorization": f"Bearer {token}"})

        # Have to check the status code because the EarthScope SDK does not raise an exception for a failed request
        if r.status_code == requests.codes.ok:
            with open(Path(Path(directory_to_save_file) / file_name), 'wb') as f:
                for data in r:
                    f.write(data)
        else:
            print(f"failure: {r.status_code}, {r.reason}")

elif (esver[0] == '1') :

    # These imports are needed for the EarthScope SDK >= 1.0.0, which uses OAuth2 authentication
    import time
    from earthscope_sdk import EarthScopeClient

    BASE_URL = "https://data.earthscope.org/archive/gnss/products/position"

    def build_csv_url(station: str, author: str = "cwu", frame: str = "igs14") -> str:
        """Builds the URL for a non-detrended GNSS position time-series CSV.

        Args:
            station: the station code in the EarthScope archive
            author: author code for the GNSS product
            frame: reference frame for the GNSS product

        Returns:
            A URL string for the CSV file

        Raises:
            ValueError: if the station code is empty or invalid

        Example:
            https://data.earthscope.org/archive/gnss/products/position/P162/P162.cwu.igs14.csv
        """
        try:
            st = station.strip().upper()
            if not st:
                raise ValueError
            return f"{BASE_URL}/{st}/{st}.{author.lower()}.{frame.lower()}.csv"
        except Exception as e:
            raise ValueError(f"Invalid station code: {station}") from e


    def fetch_csv(url: str, outpath: Path, headers: dict = None, retries: int = 3, backoff: float = 1.5) -> tuple:
        """Downloads a CSV file with repeated attempts and basic response checking.

        Args:
            url: URL of the CSV file to download
            outpath: local path where the CSV file will be saved
            headers: optional HTTP headers to include in the request
            retries: total number of download attempts
            backoff: delay multiplier between attempts, in seconds

        Returns:
            A tuple containing the URL, whether the download succeeded, and an error message.
            If the download succeeds, the error message is an empty string.
        """
        err = ""
        for attempt in range(1, retries + 1):
            try:
                r = requests.get(url, headers=headers, timeout=30)
                if r.ok:
                    outpath.parent.mkdir(parents=True, exist_ok=True)
                    outpath.write_bytes(r.content)
                    return (url, True, "")
                err = f"HTTP {r.status_code} {r.reason}"
            except requests.RequestException as e:
                err = str(e)
            time.sleep(backoff ** attempt)
        return (url, False, err)

    def download_es_file(es_client: EarthScopeClient, station: str, frame: str = "igs14", outdir: object = None) -> None:
        """Downloads a non-detrended GNSS position time-series CSV from EarthScope.

        Args:
            es_client: authenticated EarthScope client
            station: station code to download
            frame: reference frame for the GNSS product
            outdir: local directory where the CSV file will be saved
        """

        if outdir is None:
            outdir = DATA_DIR / "NGF"

        outdir = Path(outdir)
        outdir.mkdir(parents=True, exist_ok=True)

        token = es_client.ctx.auth_flow.access_token
        headers = {"Authorization": f"Bearer {token}"}

        failures = []
        url = build_csv_url(station, "cwu", frame)
        fname = Path(url).name
        outpath = outdir / fname
        url, ok, err = fetch_csv(url, outpath, headers=headers)
        if ok:
            print(f"✓ {fname}")
        else:
            print(f"✗ {fname}  --  {err}")
            failures.append((fname, err))

        if failures:
            print("\n⚠️ Some downloads failed — check author/station/frame availability.")

    print('New EarthScope_SDK version',esver,'set up')

else:
    print('Unknown earthscope_sdk version ',esver)

# Test URL to try to download a file from EarthScope; may require login, depending on EarthScope_sdk version
url = "https://gage-data.earthscope.org/archive/gnss/products/velocity/cwu.final_igs14.vel"
filename = "cwu.final_igs14.vel"

if ( NGF_test and esver[0]== '0' ) :
    print("Getting ",url)
    print("Link for login may appear below")
    get_es_file(url)

if ( NGF_test and esver[0] == '1' ):
    try:
        from earthscope_sdk import EarthScopeClient
        from earthscope_sdk.auth.auth_flow import NoAccessTokenError
        es_client = EarthScopeClient()
        token = es_client.ctx.auth_flow.access_token
        headers = {"Authorization": f"Bearer {token}"}
    except NoAccessTokenError:
        print("EarthScope authentication required. Opening the EarthScope data portal...")
        webbrowser.open("https://data.earthscope.org")
        print("If the browser login does not create a token, run `es login` in your terminal, then re-run this cell.")
        raise SystemExit("EarthScope login needed; run es login and try again.")

    outpath = Path("./"+filename)
    print('Calling fetch_csv with ',url,filename)
    url, ok, err = fetch_csv(url, outpath, headers=headers)
    if ok:
        print(f"✓ {filename}")
    else:
        print(f"✗ {filename}  --  {err}")


### Setting up Directories

In [ ]:
# TODO: Possible future improvement: additional organization support and/or user-defined organization list
orglist = ["NGF","UNR", "JPL"]

def make_if_absent(folderpath: str) -> None:
    """Creates a folder for storing data if it does not already exist.

    Args:
        folderpath: path of the folder to check or create
    """
    if not exists(folderpath): makedirs(folderpath)

make_if_absent(DATA_DIR)
for orgname in orglist:
    make_if_absent(DATA_DIR / orgname)

### Reading from Existing Data

In [ ]:
data_of = {}

for org in orglist:
    filepath = DATA_DIR / org / f"{org}_data.json"
    if exists(filepath):
        with open(filepath, "r") as data_file:
            data_of[org] = json.load(data_file)

### General Functions

In [ ]:
def number_list(text: str, default: list) -> list:
    """Reads a list or tuple of numbers entered in a widget.

    Args:
        text: text such as "(10, 10, 30)"
        default: values to return when the text is not a valid numeric list

    Returns:
        The entered values as floats, or a copy of the default values.
    """
    return core_number_list(text, default)


def filter_sigmas(df: pd.DataFrame, columns: list, limits: tuple, conversion: float = 1.0) -> pd.DataFrame:
    """Removes rows whose uncertainty is above an enabled component limit.

    Args:
        df: time-series table
        columns: North, East, and Up sigma column numbers
        limits: North, East, and Up limits; zero disables that component
        conversion: number of limit units in one table unit

    Returns:
        The filtered table.
    """
    return core_filter_sigmas(df, columns, limits, conversion)


def FitTS(xdata: np.ndarray, ydata: np.ndarray, sig: np.ndarray) -> tuple:
    """Fits a weighted linear trend to one time-series component.

    Args:
        xdata: time values in days since 2000-01-01
        ydata: position values for one component
        sig: uncertainty values for each position value

    Returns:
        A tuple containing the polynomial fit estimate, velocity estimate with uncertainty,
        and fit statistics as [WRMS, chi, number_of_data].
    """

    ## Variable Reference Table ##
        # yr: years since 2000/1/1
        # wgh: weights, determined inversely proportional to the uncertainties (sigma^2)
        # ndata: total number of data points
        # A: matrix of the normal equation coefficients, calculated as np.transpose([np.ones(ndata), yr])
        # NormEQ: matrix of the normal equations, calculated
        # Bvec: vector of the normal equations, calculated as (X.T * OmInv) @ ydata.T
        # MCov: covariance matrix of the fit parameters, calculated as the inverse of the normal equations
        # MEst: fit parameters, calculated as MCov @ Bvec
        # Res: residuals, calculated from ydata - X @ MEst


    # Set up to fit linear trend.  In later updates we could
    # create more elaborate models with offsets, periodic and post-
    # seismic componensts.
    yr = xdata/365.25
    wgh = 1./sig**2
    ndata = int(xdata.size)
    A = np.transpose([np.ones(ndata), yr])
    NormEQ = np.matmul(np.transpose(A)*wgh, A)
    Bvec = np.matmul(np.transpose(A)*wgh, np.transpose(ydata))
    MCov = np.linalg.inv(NormEQ)
    MEst = np.matmul(MCov,Bvec)
    Res = ydata-np.matmul(A,MEst)
    chi = np.sqrt(np.dot(np.transpose(Res), Res*wgh) / (ndata-2))
    wrms = np.sqrt(ndata/np.sum(wgh)) * chi
    pfe = np.flip(MEst) # Set reverse order like polyfit

    return pfe, [MEst[1],np.sqrt(MCov[1,1])], [wrms,chi,ndata]

def detrended(xdata: np.ndarray, ydata: np.ndarray, sig: np.ndarray) -> tuple:
    """Removes the weighted linear trend from one time-series component.

    Args:
        xdata: time values in days since 2000-01-01
        ydata: position values for one component
        sig: uncertainty values for each position value

    Returns:
        A tuple containing detrended residuals, velocity estimate, and fit statistics.
    """
    pfe , Vel, Stat = FitTS(xdata, ydata, sig)
    return ydata - (pfe[0] * xdata / 365.25 + pfe[1]), Vel, Stat # Stat = statistics = (WRMS, NRMS, #data)


# Use the reusable pure implementations for all later notebook calls.
FitTS = core_fit_ts
detrended = core_detrended

def remove_outliers_function(nd: np.ndarray, ed: np.ndarray, ud: np.ndarray, ns: np.ndarray, es: np.ndarray, us: np.ndarray, times: np.ndarray, td: np.ndarray, multiplier: float) -> tuple:
    """Removes points whose North, East, or Up value is far from the mean.

    Args:
        nd: North position values
        ed: East position values
        ud: Up position values
        ns: North uncertainty values
        es: East uncertainty values
        us: Up uncertainty values
        times: datetime values used for plotting
        td: time values in days since 2000-01-01
        multiplier: number of standard deviations allowed before removal

    Returns:
        a tuple containing the filtered position, uncertainty, datetime, and time arrays.
    """
    bool_list = [all(abs(lst[i]-np.mean(lst)) <= multiplier*np.std(lst) for lst in (nd, ed, ud)) for i in range(len(nd))]
    nd_new = nd[bool_list]
    ed_new = ed[bool_list]
    ud_new = ud[bool_list]
    ns_new = ns[bool_list]
    es_new = es[bool_list]
    us_new = us[bool_list]
    times_new = times[bool_list]
    td_new = td[bool_list]
    return nd_new, ed_new, ud_new, ns_new, es_new, us_new, times_new, td_new


def calc_distance_earth(start_lat: float, start_lon: float, distance_km: float, direction: str = "east") -> list:
    """Calculates a new latitude and longitude after moving along Earth's surface.

    Args:
        start_lat: starting latitude in decimal degrees
        start_lon: starting longitude in decimal degrees
        distance_km: distance to move in kilometers
        direction: direction to move; one of "east", "west", "north", or "south"

    Returns:
        A list containing the new latitude and longitude.
    """

    r = 6371.0 # radius of Earth
    if direction == "east":
        lat_rad = math.radians(start_lat)
        delta = (distance_km/(r * math.cos(lat_rad))) * (180/math.pi)
        end_lon = start_lon + delta
        end_lat = start_lat

    elif direction == "west":
        lat_rad = math.radians(start_lat)
        delta = (distance_km/(r * math.cos(lat_rad))) * (180/math.pi)
        end_lon = start_lon - delta
        end_lat = start_lat

    elif direction == "north":
        delta = distance_km / r
        lat_rad = math.radians(start_lat)
        new_lat_rad = lat_rad + delta
        end_lat = math.degrees(new_lat_rad)
        end_lon = start_lon

    elif direction == "south":
        delta = distance_km / r
        lat_rad = math.radians(start_lat)
        new_lat_rad = lat_rad - delta
        end_lat = math.degrees(new_lat_rad)
        end_lon = start_lon

    return [end_lat, end_lon]


### Timeseries Readers

In [ ]:
## FETCH TIMESERIES DATA

SigLim = (10,10,30) # standard deviation

# UNR Data Fetcher
def Read_UNR(site: str, update: bool) -> tuple:
    """Reads a UNR station time series from the local cache or UNR server.

    Args:
        site: 4-character station code
        update: whether to download a fresh copy instead of using the local file

    Returns:
        A tuple containing datetime values and a NEU time-series array.
    """
    site = site.strip().upper()
    # TODO: check if site is down first
    # MOD TAH 241223: Use IGS14 time series.
    file_directory = DATA_DIR / "UNR" / f"{site.upper()}.csv"
    file_name = site.upper()+".tenv3"
    fetch_url="https://geodesy.unr.edu/gps_timeseries/IGS20/tenv3/IGS20/"+file_name

    if exists(file_directory) and not bool(update) :
        print('Reading from file ', file_directory)
        df = pd.read_csv(file_directory, delimiter=',')
    else:
        print('Downloading from ', fetch_url)
        try:
            df = pd.read_csv(fetch_url, delimiter=r"\s+", header=1)
            df.to_csv(file_directory, index=False)
        except:
            df = []
            with timeseries_output:
                display(wg.HTML('''<em style="color:red">UNR Site '''+site.upper()+''' cant be found</em>'''))
    ninit = len(df)

    # Note: UNR stores East, North, Up sigmas in meters.
    df = filter_sigmas(df, [15, 14, 16], SigLim, 1000)
    if any(limit > 0 for limit in SigLim):
        print("Number after >{} {} {} mm NEU sigma removal".format(*SigLim),len(df),"Read",ninit)
    npdat = df.to_numpy()

    t=list(npdat[:,1])
    nd=list((npdat[:,10]-npdat[0,10])*1000) # Implemented to remove first value so it starts at zero
    # Note: the same as NGF. (Problem if times of first data point are different)
    ed=list((npdat[:,8]-npdat[0,8])*1000)
    ud=list((npdat[:,12]-npdat[0,12])*1000)
    ns=list(npdat[:,15]*1000) ; es=list(npdat[:,14]*1000) ; us=list(npdat[:,16]*1000)

    n = 0
    to = []
    td = np.zeros(len(t))
    for v in t:
        to = np.append(to, datetime.strptime(v, '%y%b%d')+timedelta(hours=12))
        dt = to[n] - datetime(2000, 1, 1,0,0)  # dt: time difference from 2000/1/1
        td[n] = dt.total_seconds()/86400.  # td: days from 2000/1/1
        n += 1

    tseries = np.array([td,nd,ns,ed,es,ud,us])
    times = to
    return times, tseries

def Read_JPL(site: str, update: bool) -> tuple:
    """Reads a JPL station time series from the local cache or JPL server.

    Args:
        site: 4-character station code
        update: whether to download a fresh copy instead of using the local file

    Returns:
        A tuple containing datetime values and a NEU time-series array.
    """
    site = site.strip().upper()
    # TODO: check if site is down first
    csv_file_directory = DATA_DIR / "JPL" / f"{site}.csv"

    fetch_url="https://sideshow.jpl.nasa.gov/pub/JPL_GPS_Timeseries/repro2018a/post/point/"+site+".series"

    if exists(csv_file_directory) and not bool(update):
        print('Reading from file',csv_file_directory)
        df = pd.read_csv(csv_file_directory,delimiter=',')
    else:
        print('Downloading from ',fetch_url)
        try:
            df = pd.read_csv(fetch_url,delimiter=r"\s+")
            df.to_csv(csv_file_directory,index=False)
        except:
            df=[]
            with timeseries_output:
                display(wg.HTML('''<em style="color:red">JPL Site '''+site.upper()+''' cant be found</em>'''))

    ninit = len(df)
    # Note: JPL stores East, North, Up sigmas in meters
    df = filter_sigmas(df, [5, 4, 6], SigLim, 1000)
    if any(limit > 0 for limit in SigLim):
        print("Number after >{} {} {} mm NEU sigma removal".format(*SigLim),len(df),"Read",ninit)

    npdat = df.to_numpy()

    nd=list((npdat[:,2]-npdat[0,2])*1000) # Implemented to remove first value so it starts at zero
    # Note: the same as NGF. (Problem if times of first data point are different)
    ed=list((npdat[:,1]-npdat[0,1])*1000) # 2nd column
    ud=list((npdat[:,3]-npdat[0,3])*1000) # 4th column
    ns=list(npdat[:,5]*1000) ; es=list(npdat[:,4]*1000) ; us=list(npdat[:,6]*1000) # what do any of these mean
    to = np.empty(len(npdat[:,10]), dtype = object)

    for i in range(len(to)):
        each_date = datetime((int(npdat[:,11][i])), int((npdat[:,12][i])), int((npdat[:,13][i])))+timedelta(hours=12)
        to[i] = each_date

    td_counter = 0
    td = np.zeros(len(npdat[:,10]))
    for time_value in npdat[:,10]:
        td[td_counter] = time_value/86400 # td: days from 2000/1/1 (file has seconds from 2000/1/1, 12:00 hr)
        td_counter += 1

    tseries = np.array([td,nd,ns,ed,es,ud,us])
    times = to
    return times, tseries




def Read_NGF(site: str, update: bool) -> tuple:
    """Reads an NGF station time series from the local cache or EarthScope server.

    Args:
        site: 4-character station code
        update: whether to download a fresh copy instead of using the local file

    Returns:
        A tuple containing datetime values and a NEU time-series array.
    """
    global SigLim
    site = site.strip().upper()
    filen = site.upper()+".cwu.igs14.csv"
    filepath = DATA_DIR / "NGF" / filen
    whurl="https://data.unavco.org/archive/gnss/products/position/"+site.upper()+"/"+filen

    #PBO Station Position Time Series.
    #Format Version, 1.2.0
    #Reference Frame, igs14
    #4-character ID, P177
    #Station name, CoDeTierraCN2008
    #Begin Date, 2008-05-09
    #End Date, 2021-10-06
    #Release Date, 2021-10-07
    #Source file, P177.cwu.igs14.pos
    #offset from source file, 166.01 mm North, -139.45 mm East, -3.62 mm Vertical
    #Reference position, 37.5281683684 North Latitude, -122.4950535711 East Longitude, 71.79172 meters elevation
    #Date, North (mm), East (mm), Vertical (mm), North Std. Deviation (mm), East Std. Deviation (mm), Vertical Std. Deviation (mm), Quality,
    #2008-05-09,0.00, 0.00, 0.00, 2.08, 1.66, 7.84, repro,
    #2008-05-10,0.38, 0.83, 2.66, 2.13, 1.7, 7.99, repro,
    #...
    if exists(filepath) and not bool(update):
        print('Reading from file',filen)
        df = pd.read_csv(filepath,delimiter=',',header=11)
    else:
        if ( esver[0] == '0' ) :
            print('Downloading from ',whurl)
            try:
                get_es_file(whurl)
                df = pd.read_csv(filepath,delimiter=',',header=11)
            except:
                df = []
                with timeseries_output:
                    display(wg.HTML('''<em style="color:red">NGF Site '''+site.upper()+''' cant be found</em>'''))

        else:
            # Uses ES 1.0 instead of ES 0.0.1, which is now obsolete.
            print('Getting',site.upper(),'with EarthSopeClient')
            es = EarthScopeClient()
            # Refreshes access token if necessary; if failed, use the EarthScope CLI again to login: es login
            # Example error: "NoRefreshTokenError: No refresh token was found. Please re-authenticate."
            es.ctx.auth_flow.refresh_if_necessary()
            download_es_file(es, site.upper(), frame="igs14")

        df = pd.read_csv(filepath,delimiter=',',header=11)

    ninit = len(df)
    # Note: NGF stores its sigma columns in millimeters already.
    df = filter_sigmas(df, [4, 5, 6], SigLim)
    if any(limit > 0 for limit in SigLim):
        print("Number after >{} {} {} mm NEU sigma removal".format(*SigLim),len(df),"Read",ninit)
    npdat = df.to_numpy()

    t=list(npdat[:,0]);nd=list(npdat[:,1]); ed=list(npdat[:,2]); ud=list(npdat[:,3]);
    ns=list(npdat[:,4]) ; es=list(npdat[:,5]) ; us=list(npdat[:,6])

    n = 0
    to = []
    td = np.zeros(len(t))
    for v in t:
        # Note: moves time to middle of the day because the data is recorded at 12:00 UTC.
        to = np.append(to,datetime.strptime(v, '%Y-%m-%d')+timedelta(hours=12))
        dt = to[n] - datetime(2000, 1, 1)  # Time difference from 2000/1/1
        td[n] = dt.total_seconds()/86400.  # Days from 2000/1/1
        n += 1

    tseries =  np.array([td,nd,ns,ed,es,ud,us])
    times = to
    return times, tseries

Read = {
        "NGF": Read_NGF,
        "UNR": Read_UNR,
        "JPL": Read_JPL
    }

### Map Functions

In [ ]:
d_km = lambda coords1, coords2: geopy.distance.geodesic(coords1,coords2).km
d_mi = lambda coords1, coords2: geopy.distance.geodesic(coords1,coords2).miles

def add_map_layer(map_obj: Map, layer: object) -> None:
    map_obj.add(layer)


def plot_locations(map_obj: Map, coordslist: list, tooltiplist: list, color: str = "blue", dotrad: int = 3) -> None:
    """Plots simple dot markers on an ipyleaflet map.

    Args:
        map_obj: ipyleaflet map object
        coordslist: list of latitude and longitude pairs
        tooltiplist: list of text labels for the marker popups
        color: marker color
        dotrad: marker radius in pixels
    """
    for coords, tooltip in zip(coordslist, tooltiplist):
        marker = CircleMarker(
            location=coords,
            radius=dotrad,
            color=color,
            weight=0,
            fill_color=color,
            fill_opacity=1,
            opacity=1,
        )
        marker.popup = HTML(value=str(tooltip))
        add_map_layer(map_obj, marker)



def map_station_summary(siteid: str, org: str) -> str:
    """Builds a short station summary for a map popup.

    Args:
        siteid: 4-character station code
        org: data source name

    Returns:
        A formatted text summary of the station data.
    """
    try:
        info = data_of[org][siteid]
        loc = info.get("location", [None, None])
        hgt = info.get("height", "")
        vel = info.get("velocity", "No velocity")
        sig = info.get("velsig", "")
        return (
            f"Site: {siteid}\n"
            f"Source: {org}\n"
            f"Lat/Lon: {loc[0]}, {loc[1]}\n"
            f"Height: {hgt}\n"
            f"Velocity NEU: {vel}\n"
            f"Sigma NEU: {sig}"
        )
    except Exception:
        return f"{siteid} ({org})"


def add_map_site_to_timeseries(siteid: str, org: str, plot_now: bool = False) -> None:
    """Adds a station selected from the map to the time-series widget.

    Args:
        siteid: 4-character station code
        org: data source name
        plot_now: whether to immediately plot the selected station
    """
    # Note: this was implemented to let the map talk directly to the time-series widget.
    extendedsiteid = siteid + " (" + org + ")"
    if 'ts_site_form' in globals():
        ts_site_form.value = siteid
    if 'org_ts_select' in globals() and org in org_ts_select.options:
        org_ts_select.value = org
    if 'ts_sites' in globals() and extendedsiteid not in ts_sites.options:
        ts_sites.options = list(ts_sites.options) + [extendedsiteid]
    if 'ts_sites' in globals():
        ts_sites.value = (extendedsiteid,)
    if plot_now and 'list_to_graph' in globals():
        list_to_graph(None)


def station_popup(siteid: str, org: str, tooltip: str) -> VBox:
    """Builds the ipywidgets popup shown when a station marker is clicked.

    Args:
        siteid: 4-character station code
        org: data source name
        tooltip: station text to show in the popup

    Returns:
        A VBox widget containing station text, buttons, and copyable data.
    """
    # ipyleaflet popups can contain real widgets, not just text.
    add_button = wg.Button(description="Add to Timeseries")
    plot_button = wg.Button(description="Plot Now", button_style="success")
    data_box = wg.Textarea(
        value=map_station_summary(siteid, org),
        rows=5,
        disabled=False,
        layout=wg.Layout(width="calc(100% - 4px)", margin="0 2px 0 0"),
    )

    def add_clicked(_button):
        add_map_site_to_timeseries(siteid, org, plot_now=False)

    def plot_clicked(_button):
        add_map_site_to_timeseries(siteid, org, plot_now=True)

    add_button.on_click(add_clicked)
    plot_button.on_click(plot_clicked)

    return VBox([
        HTML(value=f"<b>{siteid}</b> ({org})"),
        HBox([add_button, plot_button]),
        data_box,
    ])


def plot_site_locations(map_obj: object, org: str, coordslist: list, siteidlist: list, tooltiplist: list, color: str = "blue", dotrad: int = 3) -> None:
    """Plots station markers with station-aware popups on an ipyleaflet map.

    Args:
        map_obj: ipyleaflet map or layer group
        org: data source name
        coordslist: list of latitude and longitude pairs
        siteidlist: list of station codes
        tooltiplist: list of popup labels
        color: marker color
        dotrad: marker radius in pixels
    """
    # Attach widgets directly to markers, like the demo. This lets Leaflet size
    # the popup from the real controls instead of constraining a widget inside a Popup layer.
    for coords, siteid, tooltip in zip(coordslist, siteidlist, tooltiplist):
        marker = CircleMarker(
            location=coords,
            radius=dotrad,
            color=color,
            weight=0,
            fill_color=color,
            fill_opacity=1,
            opacity=1,
        )

        # Build the demo-style direct marker widget only after first hover or click.
        def load_popup(_event=None, marker=marker, siteid=siteid, org=org, tooltip=tooltip, **_kwargs):
            """Create the direct marker popup when the marker is first interacted with.

            Args:
                _event: Marker hover or click event payload.
                **_kwargs: Additional event fields.
            """
            if marker.popup is None:
                marker.popup = station_popup(siteid, org, tooltip)

        marker.on_mouseover(load_popup)
        marker.on_click(load_popup)
        add_map_layer(map_obj, marker)

def nearby_sites(org: str, site: object, radius_in_km: float) -> tuple:
    """Finds stations within a given distance of a site or coordinate pair.

    Args:
        org: data source name
        site: station code or coordinate pair
        radius_in_km: search radius in kilometers

    Returns:
        A tuple containing nearby station records and the center coordinate pair.
    """
    if isinstance(site, str):
        site = site.strip()
        if site[0] in "([": # if it's a coordinate pair instead of a site id
            coords = ast.literal_eval(site)
            if not isinstance(coords, (list, tuple)) or len(coords) != 2:
                raise ValueError("Coordinates must contain latitude and longitude")
            site = [float(value) for value in coords]
        else:
            site = site.upper()

    cur_coords = site

    if isinstance(site,str):
        cur_coords = data_of[org][site]["location"]

    assert not isinstance(cur_coords[0],str) # by now cur_coords is a coordinate pair regardless of input

    id_coord_dist_list = []

    for siteid in data_of[org]:
        try:
            site_coords = data_of[org][siteid]["location"]
            distance = d_km(site_coords, cur_coords)
            if distance < radius_in_km : #  and site != siteid:  (siteid site as well for tooltip)
                id_coord_dist_list.append((siteid,*site_coords,distance))
        except: pass

    return id_coord_dist_list, cur_coords


def basic_circle(map_obj: object, center: list, radius_in_km: float) -> None:
    """Draws a radius circle around a center point on the map.

    Args:
        map_obj: ipyleaflet map or layer group
        center: center latitude and longitude pair
        radius_in_km: circle radius in kilometers
    """
    circle = Circle(
        location=center,
        radius=radius_in_km * 1000, # in metres
        color="black",
        weight=1,
        fill_opacity=0.0,
        opacity=1,
        fill_color="green",
        fill=True,
    )

    add_map_layer(map_obj, circle)


def vec_add(c1,c2):
    return core_vec_add(c1, c2)

def vec_sub(c1,c2):
    return core_vec_sub(c1, c2)

def velocity_endpoint(center: list, direction: list, scale: float) -> list:
    """Calculates the map endpoint for a North/East velocity vector.

    Args:
        center: starting latitude and longitude pair.
        direction: velocity vector as North, East, and Up components.
        scale: map scale in kilometers per millimeter per year.

    Returns:
        A latitude and longitude pair for the vector endpoint.
    """
    return core_velocity_endpoint(center, direction, scale)


def draw_vector(map_obj: object, center: list, direction: list, scale: float, color: str = "black", label: str = "None") -> None:
    """Draws a horizontal velocity vector with an arrowhead on the map.

    Args:
        map_obj: ipyleaflet map or layer group
        center: starting latitude and longitude pair
        direction: velocity vector as [north, east, up]
        scale: map scale in kilometers per mm/yr
        color: line color
        label: optional text label to place at the vector endpoint
    """

    endpoint = velocity_endpoint(center, direction, scale)

    line = Polyline(locations=[center, endpoint], weight=2, color=color)
    add_map_layer(map_obj, line)

    north, east = direction[0], direction[1]
    length_km = math.hypot(north, east) * scale
    if length_km > 0:
        unit_north, unit_east = north / math.hypot(north, east), east / math.hypot(north, east)
        head_length = length_km * 0.18
        head_half_width = head_length * 0.5
        back_north, back_east = -unit_north * head_length, -unit_east * head_length
        side_north, side_east = -unit_east * head_half_width, unit_north * head_half_width
        left_tip = velocity_endpoint(endpoint, [back_north + side_north, back_east + side_east], 1)
        right_tip = velocity_endpoint(endpoint, [back_north - side_north, back_east - side_east], 1)
        arrowhead = Polyline(locations=[left_tip, endpoint, right_tip], weight=2, color=color)
        add_map_layer(map_obj, arrowhead)

    if label != "None":
        lab = Marker(
            location=endpoint,
            icon=DivIcon(
                html='<div style="font-size: 12pt; white-space: nowrap">'+label+'</div>',
                icon_size=(250,36),
                icon_anchor=(0,0),
            )
        )
        add_map_layer(map_obj, lab)


### Timeseries Functions

In [ ]:
def remove_brac(string: str) -> tuple:
    """Splits a display label into station code and data source.

    Args:
        string: label formatted like "P123 (UNR)"

    Returns:
        A tuple containing the station code and data source.
    """
    return core_remove_brac(string)


def plot_ts_graph(siteid: str, org: str, Update: bool, detrend: bool, ax0: object, ax1: object, ax2: object, yearrange: list, breaks: bool, errorbars: list, errorbar_outline: list, outlier: float, shift: float, color: str) -> object:
    """Adds one station time series to the three-component plot.

    Args:
        siteid: 4-character station code
        org: data source name
        Update: whether to download a fresh copy of the time series
        detrend: whether to remove the fitted linear trend
        ax0: matplotlib axis for North displacement
        ax1: matplotlib axis for East displacement
        ax2: matplotlib axis for Up displacement
        yearrange: start and end datetime values for the plot
        breaks: whether to remove known break offsets
        errorbars: error bar settings
        errorbar_outline: filled error outline settings
        outlier: outlier removal multiplier, or 0 to skip removal
        shift: vertical display shift to apply to this station
        color: plot color

    Returns:
        A DataFrame of fitted velocity statistics when detrending is enabled; otherwise an empty list.
    """

    try:
        times, tseries = Read[org](siteid, Update)
    except:
        print('Error getting',siteid,'from',org)
        times = []; tseries = []
        return

    td, nd, ns, ed, es, ud, us = tseries

    keep = np.array([yearrange[0] <= time <= yearrange[1] for time in times])
    times, td = np.asarray(times)[keep], td[keep]
    nd, ns = nd[keep], ns[keep]
    ed, es = ed[keep], es[keep]
    ud, us = ud[keep], us[keep]
    if len(times) == 0:
        print("No data for", siteid, "in the selected date range")
        return []

    if breaks:
        if "breaks" in data_of[org][siteid]:
            breaktimes_dict = {}
            for dt in data_of[org][siteid]["breaks"]:
                try:
                    yr_mm_day = [int(date) for date in dt[1:-1].split(",")][0:5]
                    breaktime = datetime(*yr_mm_day)
                    breaktimes_dict[breaktime] = [data_of[org][siteid]["breaks"][dt]["offsets"], int(np.where(times >= breaktime)[0][0])]
                except:
                    pass
            # [45.01, 0.67, 39.72, 0.59, -27.67, 1.83], [dN (mm), sN (mm), dE (mm), sE (mm), dU (mm), sU (mm)]
            interval_counter = 0
            breaktimes_list = list(breaktimes_dict)
            ndat = len(ud)
            while interval_counter in range(len(breaktimes_dict)):
                dict_key = breaktimes_list[interval_counter]
                interval_start = breaktimes_dict[dict_key][1]
                # Using -1 as last point skipped last point
                offsets = breaktimes_dict[dict_key][0]
                nd[interval_start:ndat] = nd[interval_start:ndat] - offsets[0]
                ed[interval_start:ndat] = ed[interval_start:ndat] - offsets[2]
                ud[interval_start:ndat] = ud[interval_start:ndat] - offsets[4]
                ns[interval_start:ndat] = np.hypot(ns[interval_start:ndat], offsets[1])
                es[interval_start:ndat] = np.hypot(es[interval_start:ndat], offsets[3])
                us[interval_start:ndat] = np.hypot(us[interval_start:ndat], offsets[5])
                interval_counter +=1
        else:
            print("No breaks information!")

    dfa = []
    if detrend:
        nd, VelN, StatN = detrended(td, nd, ns)
        ed, VelE, StatE = detrended(td, ed, es)
        ud, VelU, StatU = detrended(td, ud, us)

        if outlier: # while length of the array before and after are different, keep iterating the code
            initial_length = len(nd)
            length1 = 1
            length2 = 2
            iteration_cnt = 0
            while length1 != length2:
                length1 = len(nd)
                nd, ed, ud, ns, es, us, times, td = remove_outliers_function(nd, ed, ud, ns, es, us, times, td, outlier)
                nd, dVelN, StatN = detrended(td, nd, ns)
                VelN[0] = VelN[0]+dVelN[0]   # Fit is to residuals, so update total

                ed, dVelE, StatE = detrended(td, ed, es)
                VelE[0] = VelE[0]+dVelE[0]   # Fit is to residuals, so update total
                ud, dVelU, StatU = detrended(td, ud, us)
                VelU[0] = VelU[0]+dVelU[0]   # Fit is to residuals, so update total

                length2 = len(nd)
                iteration_cnt +=1
            print(f"Final {len(nd)} data; Number of outliers removed: {initial_length - len(nd)} with {iteration_cnt} iterations: ")

        VelNEU = VelN+VelE+VelU ; StatNEU = StatN[0:2]+StatE[0:2]+StatU
        labl = str(siteid)+'-'+str(org)
        dfa = pd.DataFrame([VelNEU+StatNEU], \
            columns=['Vn','σVn','Ve','σVe','Vu','σVu','WRMS N','χn','WRMS E','χe','WRMS U','χu','Num'],index=[labl])


    nd = nd+shift ;  ed = ed+shift ; ud = ud+shift*3

    if errorbars[0]:
        ax0.errorbar(times,nd,yerr=[ns,ns],errorevery=errorbars[1], capsize=2, color = color, linewidth=1) #, ecolor='black')
        ax1.errorbar(times,ed,yerr=[es,es],errorevery=errorbars[1], capsize=2, color = color, linewidth=1) #, ecolor='black')
        ax2.errorbar(times,ud,yerr=[us,us],errorevery=errorbars[1], capsize=2, color = color, linewidth=1) #, ecolor='black')

    if errorbar_outline[0]:
        for ax, d, s in [(ax0, nd, ns), (ax1, ed, es), (ax2, ud, us)]:
            ax.fill(list(times) + list(reversed(times)),
                list(d+s) + list(np.flip(d-s)),
                alpha=errorbar_outline[1], linewidth=1, color = color, label="_"+siteid)

    for ax, d in [(ax0, nd), (ax1, ed), (ax2, ud)]:
        ax.plot(times, d, linewidth=0.5, label=siteid + " (" + org + ")", color = color)
        if "breaks" in data_of[org][siteid]:
            for dt in data_of[org][siteid]["breaks"]:
                yr_mm_day = [int(date) for date in dt[1:-1].split(",")][0:3]
                breaktime = datetime(*yr_mm_day)
                try:
                    np.where(times >= breaktime)[0][0]
                    ax.axvline(x=breaktime, color='r', ls='--')
                except:
                    pass

    return dfa

def plot_ts_graph_list(idlist: list, Update: bool, yearrange: list, breaks: bool, detrend: bool, errorbars: list, errorbar_outline: list, outlier: float, resolution: str = "Low Res", shift: object = 0) -> None:

    """Plots North, East, and Up time series for a list of selected stations.

    Args:
        idlist: selected station labels formatted like "P123 (UNR)"
        Update: whether to download fresh time-series files
        yearrange: start and end datetime values for the plot
        breaks: whether to remove known break offsets
        detrend: whether to remove fitted linear trends
        errorbars: error bar settings
        errorbar_outline: filled error outline settings
        outlier: outlier removal multiplier, or 0 to skip removal
        resolution: figure resolution setting
        shift: dictionary of display shifts by station label
    """

    global tsfig

    tsfig = plt.figure(figsize=(15, 10), dpi = {"HD": 600, "Regular": 300, "Low Res": 100}[resolution])
    gs = gridspec.GridSpec(3, 1, height_ratios=[1, 1, 1])

    ax0 = plt.subplot(gs[0])
    ax0.set_ylabel("ΔNorth (mm)")
    ax0.yaxis.set_tick_params(labelrotation=90)
    plt.setp(ax0.get_xticklabels(), visible=False)

    ax1 = plt.subplot(gs[1], sharex = ax0)
    ax1.set_ylabel("ΔEast (mm)")
    ax1.yaxis.set_tick_params(labelrotation=90)
    plt.setp(ax1.get_xticklabels(), visible=False)

    ax2 = plt.subplot(gs[2], sharex = ax0)
    ax2.set_ylabel("ΔUp (mm)")
    ax2.yaxis.set_tick_params(labelrotation=90)
    ax2.set_xlabel("Time")

    colors = ["b", 'g', 'r', 'c', 'm', 'y', 'k', 'sienna']
    color_count = 0
    for idorg in idlist:
        dfl = plot_ts_graph(*remove_brac(idorg), Update, detrend, ax0, ax1, ax2, yearrange, breaks, errorbars, errorbar_outline, outlier, shift[idorg] if idorg in shift else 0, colors[color_count])
        if color_count < 8:
            color_count += 1
        else:
            color_count = 0

        if detrend:
            if color_count == 1:
                dfall = dfl
            else:
                dfall = pd.concat([dfall, dfl])

    if detrend:
        display(dfall.style.format({'Vn': '{:.2f}','σVn': '{:.3f}',
                                'Ve': '{:.2f}','σVe': '{:.3f}',
                                'Vu': '{:.2f}','σVu': '{:.3f}',
                                'WRMS N': '{:.2f}','χn': '{:.2f}',
                                'WRMS E': '{:.2f}','χe': '{:.2f}',
                                'WRMS U': '{:.2f}','χu': '{:.2f}','Num':'{:d}'}))


    leg = ax0.legend()

    plt.subplots_adjust(hspace=.0)

    if mpl.get_backend() == 'nbagg':
        from IPython.display import display as ipdisplay
        ipdisplay(tsfig)
    else:
        plt.show()


### Widgets

#### Fetcher Widgets

In [ ]:
# Buttons
update_UNR_butt     = wg.Button(description="Latest UNR", icon = "download")
update_JPL_butt     = wg.Button(description="Latest JPL", icon = "download")
update_NGF_butt    = wg.Button(description="Latest NGF", icon = "download")
breaks_select   = wg.Dropdown(options=orglist, value='UNR', distabled = False, layout=wg.Layout(width='100px'))
add_breaks_site_button = wg.Button(description="Add Breaks")
remove_breaks_site_button = wg.Button(description="Remove Breaks")
# Outputs
fetcher_output      = wg.Output()
breaks_update_output = wg.Output()

#### Availability Widgets

In [ ]:
# Inputs
site_searchbar      = wg.Text(value=None,placeholder='Site ID',description='Check Availability:',disabled=False, style= {'description_width': 'initial'})
org_avail_select    = wg.Dropdown(options=orglist, value='NGF', disabled=False, layout=wg.Layout(width='100px'))
site_search_submit  = wg.Button(description="Search!", icon = "search")
clear_log_butt      = wg.Button(description="Clear Log", icon = "times")
# Style
site_search_submit.style.button_color = 'rgb(196,253,196)'
clear_log_butt.style.button_color = 'mistyrose'
# Outputs
availability_output = wg.Output()

#### Map Widgets

In [ ]:
## TODO: vector controls changed to sliders for easier scale adjustment
# Inputs
map = ["Not initialized yet!"] # allow direct editing of the map via entry mutation
map_dynamic_layers = [None] # stations, circles, and vectors that can be replaced without remounting the map
map_reloading = [False] # prevent a second reload while the current one is still building layers
map_loading = HTML(
    value='<div style="background:rgba(255,255,255,0.92); border:1px solid #888; padding:8px 12px; font-weight:600">Updating map...</div>',
    layout=wg.Layout(display='none')
)
layout              = lambda w: wg.Layout(width=w, height='40px')
new_map_butt        = wg.Button(description="Clear / New Map", icon = "map")
reload_map_butt     = wg.Button(description="Reload Map", icon = "refresh")
close_map_butt      = wg.Button(description="Close Map", icon = "times")
station_field_layout = wg.Layout(flex='1 1 0')
site_id         = wg.Text(value=None,placeholder='e.g. P049',disabled=False, layout=station_field_layout)
org_map_select      = wg.Dropdown(options=orglist+['other'], value='NGF', disabled=False, layout=station_field_layout)
site_radius         =site_radius = wg.Text(
    value='',
    placeholder='0',
    disabled=False,
    layout=station_field_layout
)
vel_siglim_form     = wg.Text(
    value=None, description='Deviation Limit (σ):', placeholder='(1,1,2)',
    style={'description_width': 'initial'}, layout=wg.Layout(width='88%')
)
plot_vec_check      = wg.ToggleButton(description="Plot Velocities", value=False, icon='long-arrow-right', layout = layout("200px"))
velocity_mode_select = wg.Dropdown(
    options=[('Absolute', 'source'), ('Relative', 'relative')],
    value='source', disabled=False, layout=wg.Layout(width='68%')
)
arrow_length_input    = wg.FloatSlider(description = "Arrow mm/yr", min = 1, max = 50, step = 1, value = 10,
                                      readout_format = '.0f', layout = wg.Layout(width='82%'),
                                      tooltip="Length of arrow shown for scale")
velocity_scale_input   = wg.FloatSlider(description = "Vel Scale", min = 1, max = 30, step = 1, value = 10,
                                       readout_format = '.0f', layout=wg.Layout(width='82%'),
                                       tooltip='Scale from mm/yr to km on map')
colorscalefactor_input = wg.FloatSlider(description = "±U Rate", min = 1, max = 20, step = 1, value = 5,
                                       readout_format = '.0f', layout=wg.Layout(width='82%'),
                                       tooltip='±<Range for Vertical Rate> (color saturates after this value) red (negative) to blue (positive)')
thin_velocities_input  = wg.IntSlider(description = "Decimate", min = 1, max = 50, step = 1, value = 1,
                                     layout=wg.Layout(width='82%'))

# map radius lists
map_radius_list     = wg.SelectMultiple(options=[], value=[], description = "Site/Radius:", layout=wg.Layout(display='none'))
bulk_selection_input = wg.Text(
    placeholder='e.g. 1, 2, [4, 8], 12',
    layout=wg.Layout(width='78%')
)
select_bulk_sites_butt = wg.Button(description='Select', layout=wg.Layout(width='20%'))
select_all_sites_butt = wg.Button(description='Select All', layout=wg.Layout(width='100%'))
site_table          = VBox(
    [],
    layout=wg.Layout(margin='2% 0 0 0')
)
neighbor_table = VBox(
    [],
    layout=wg.Layout(margin='0 0 2% 0')
)
table_spacer        = wg.Box([], layout=wg.Layout(
    height='20px', min_height='20px', max_height='20px', flex='0 0 20px'
))
map_site_checks     = {}
add_sites_map_button  = wg.Button(description = "Add Site", icon='plus-circle')
table_action_layout = wg.Layout(width='210px', height='40px')
remove_sites_map_button = wg.Button(description = "Remove Site(s)", icon='minus-circle', layout=table_action_layout)
clear_map_list_butt     = wg.Button(description="Clear List", icon = "times", layout=table_action_layout)

site_circle_submit  = wg.Button(description="Plot Site(s)", icon = "map-marker", layout=table_action_layout)

# Style
new_map_butt.style.button_color = 'rgb(196,253,196)'
reload_map_butt.style.button_color = 'oldlace'
close_map_butt.style.button_color = 'mistyrose'
clear_map_list_butt.style.button_color = 'mistyrose'
site_circle_submit.style.button_color = 'paleturquoise'
add_sites_map_button.style.button_color = 'rgb(196,253,196)'
select_all_sites_butt.style.button_color = 'lightgoldenrodyellow'

# Basemap options
BaseMap_butt = wg.Dropdown(
    options=['OpenTopo', 'OpenStreetMap', 'ArcGIS Image'],
    value='OpenTopo',
    disabled=False,
    layout=wg.Layout(flex='1 1 0')
)
BaseMap_Set = wg.Button(description = "Set Base Map")


# Outputs
map_output          = wg.Output()
nearest_site_output = wg.Output()
graph_output        = wg.Output()

#### Timeseries Widgets

In [ ]:
# Inputs
ts_site_form        = wg.Text(value=None,placeholder='Site ID',disabled=False, style= {'description_width': 'initial'}, layout=layout("150px"))
org_ts_select       = wg.Dropdown(options=orglist+['other'], value='NGF', disabled=False, layout=wg.Layout(width='100px'))
append_butt         = wg.Button(description="Add to List", icon = "plus-square")
plot_ts_butt        = wg.Button(description="Plot", icon = "line-chart")
plot_ts_res         = wg.Dropdown(options=['HD', 'Regular', 'Low Res'], value='Low Res', disabled=False, layout=wg.Layout(width='100px'))
close_ts_butt       = wg.Button(description="Close Graph", icon = "times")
clear_list_butt     = wg.Button(description="Clear List", icon = "times")
ts_sites            = wg.SelectMultiple(options=[], value=[], description='Site List:')

# TS customizations

error_bar_check     = wg.Checkbox(description="Error Bars", value=False)
error_bar_outline_check = wg.Checkbox(description = "Error Bar Outlines", value = False)

detrend_check       = wg.Checkbox(description="Detrend", value=False)
live_update_check   = wg.Checkbox(description="Live Update", value=False)
start_year_form     = wg.Text(value=None,placeholder='YYYY-MM-DD',description='Start:',disabled=False, style= {'description_width': 'initial'}, layout=layout("150px"))
end_year_form       = wg.Text(value=None,placeholder='YYYY-MM-DD',description='End:',disabled=False, style= {'description_width': 'initial'}, layout=layout("150px"))
siglim_form         = wg.Text(value=None,placeholder='(10,10,30)',description='SigLim:',disabled=False, style= {'description_width': 'initial'}, layout=layout("150px"))

shift_value        = wg.FloatText(value=None,placeholder='0',description="Shift Values (mm)",disabled=False, style= {'description_width': 'initial'}, layout=layout("200px"))
remove_site_button  = wg.Button(description="Remove Site", icon = "minus-square")
update_customization = wg.Button(description="Update shift", icon = "arrow-up")
shift_output       = wg.Output()

thin_error_bars    = wg.IntText(description = "NErrBar", value = 1, layout=layout("150px"),tooltip='Show every Nth error bar')
error_bar_opacity  = wg.FloatText(description = "EB Opacity", layout=layout("150px"), value = 0.1)
add_breaks_ts_button = wg.Button(description = "Copy Break Data")
remove_breaks_ts_button = wg.Button(description="Clear Break Data")

remove_outliers = wg.Dropdown(description = "Remove N σ", options=[None, 1, 2, 3, 4, 5, 6], value = None, disabled = False, layout=wg.Layout(width='150px'))
show_breaks_data_button = wg.Button(description = "Show Breaks Data")
remove_breaks_checkbox = wg.Checkbox(description = "Remove Breaks", value = False)
breaks_data_output = wg.Output()

# ID points option (only in iterative mode)
id_button =  wg.Button(description = "ID points",tooltip='right-click/return to end\nmiddle/delete to remove')

# Style
append_butt.style.button_color = 'rgb(196,253,196)'
plot_ts_butt.style.button_color = 'paleturquoise'
close_ts_butt.style.button_color = 'mistyrose'
clear_list_butt.style.button_color = 'mistyrose'
remove_site_button.style.button_color = 'mistyrose'
id_button.style.button_color = 'springgreen'

# Graphics Backends
backend_butt = wg.RadioButtons(
    options=['Inline', 'ipympl', 'Interactive'],
    value='Inline',
    description='Backend Choice:',
    disabled=False
)
backend_activate = wg.Button(description = "Backend Activate")

# Outputs
timeseries_output = wg.Output()

### Widget Functions

#### Database Fetcher

In [ ]:
## TODO: NGF update auth branch now opens EarthScope portal when needed
def UpdateUNR(_):
    with fetcher_output:
        print("Downloading Data from UNR...")

    # Updated to file location URL
    plates_site = "https://geodesy.unr.edu/gps_timeseries/Plates/sta_frames.txt"
    coords_site = "https://geodesy.unr.edu/NGLStationPages/llh.out"
    # Updated to IGS14 TAH 260330.
    vels_site   = "https://geodesy.unr.edu/velocities/midas.IGS14.txt"

    ## TODO: Explore replacing with decode to avoid weird formatting issues if website changes
    plates_list = str(urllib.request.urlopen(plates_site).read())[2:].split(" \\n")[:-1]
    coords_list = str(urllib.request.urlopen(coords_site).read())[2:].split(" \\n")[:-1]

    for_json = {}

    for siteplate in plates_list:
        siteplates = [el for el in siteplate.split(" ") if el]
        for_json.setdefault(siteplates[0], {'location': None, 'height': [], 'regions': []})
        for_json[siteplates[0]]['regions'] = siteplates[1:]

    for sitecoord in coords_list:
        siteid, lon, lat, height = (el for el in sitecoord.split(" ") if el)
        for_json.setdefault(siteid, {'location': None, 'height': [], 'regions': []})
        for_json[siteid]['location'] = [float(lon), float(lat)]
        for_json[siteid]['height'] = float(height)

    ## TODO: Add reading UNR velocities
    df = pd.read_csv(vels_site,delimiter=r"\s+",usecols=[0,8,9,10,11,12,13])
    nvel = len(df) ; print("Processing",nvel,"lines")
    for i in range(nvel):
        siteid = df.iloc[i,0]
        for_json.setdefault(siteid, {})
        for_json[siteid]["velocity"] = list(df.iloc[i,[2,1,3]]*1000) # switch to mm and from ENU to NEU
        for_json[siteid]["velsig"] = list(df.iloc[i,[5,4,6]]*1000)


    global data_of

    data_of["UNR"] = for_json

    with open(DATA_DIR / 'UNR' / 'UNR_data.json', 'w') as f:
        json.dump(for_json, f)

    with fetcher_output:
        print("Latest UNR Data Downloaded")

update_UNR_butt.on_click(UpdateUNR)


def UpdateJPL(_):
    with fetcher_output:
        print("Downloading Data from JPL...")

    coords_site = "https://sideshow.jpl.nasa.gov/post/tables/table2.html"
    x = str(urllib.request.urlopen(coords_site).read()).split("\\n")
    sitecoordlist = [[xxx for xxx in xx.split(" ") if xxx]
                     for xx in x if "POS" in xx]
    sitevellist = [[xxx for xxx in xx.split(" ") if xxx]
                     for xx in x if "VEL" in xx]
    # [['AB01', 'POS', '52.2095', '-174.2048', '25492.217', '0.034', '0.024', '0.098'], ...]
    for_json = {}
    for element in sitecoordlist:
        for_json[element[0]] = {'location': [float(element[2]),float(element[3])],'height': float(element[4])/1000.0}
    for element in sitevellist:
        for_json[element[0]]['velocity'] = [float(element[2]),float(element[3]), float(element[4])]
        for_json[element[0]]['velsig'] = [float(element[5]),float(element[6]), float(element[7])]

    global data_of
    data_of["JPL"] = for_json

    with open(DATA_DIR / 'JPL' / 'JPL_data.json', 'w') as f:
        json.dump(for_json, f)

    with fetcher_output:
        print("Latest JPL Data Downloaded")

update_JPL_butt.on_click(UpdateJPL)


def UpdateNGF(_):
    with fetcher_output:
        print("Downloading Data from NGF...")

    coords_site = "https://www.unavco.org/instrumentation/networks/status/data/geoJSON/network-monitoring"

    x = json.loads(urllib.request.urlopen(coords_site).read())
    print('Back from json.load ')
    # [['AB01', 'POS', '52.2095', '-174.2048', '25492.217', '0.034', '0.024', '0.098'], ...]
    for_json = {}
    for element in x['features']:
        for_json[element["id"]] = {
            'location': [element['geometry']['coordinates'][1],element['geometry']['coordinates'][0]],
            'height': float(element['properties']['elev']),
            'region': element['properties']['region'],
            'stntype': element['properties']['stntype']}

    # Read Breaks #
    if ( esver[0] == 0 ):
        get_es_file("https://gage-data.earthscope.org/archive/gnss/products/offset/cwu.kalts_nam14.off")
    else:
        try:
            from earthscope_sdk import EarthScopeClient
            from earthscope_sdk.auth.auth_flow import NoAccessTokenError
            es_client = EarthScopeClient()
            token = es_client.ctx.auth_flow.access_token
            headers = {"Authorization": f"Bearer {token}"}
        except NoAccessTokenError:
            print("EarthScope authentication required. Opening the EarthScope data portal...")
            webbrowser.open("https://data.earthscope.org")
            print("If the browser login does not create a token, run `es login` in your terminal, then re-run this cell.")
            return
        fname = "cwu.kalts_nam14.off"
        outpath = DATA_DIR / "NGF" / "cwu.kalts_nam14.off"
        url = "https://gage-data.earthscope.org/archive/gnss/products/offset/cwu.kalts_nam14.off"
        print('Calling fetch_csv with ',url,fname)
        url, ok, err = fetch_csv(url, outpath, headers=headers)
        if ok:
            print(f"✓ {fname}")
        else:
            print(f"✗ {fname}  --  {err}")

    with open(DATA_DIR / "NGF" / "cwu.kalts_nam14.off", "r") as f:
        data = list(f)

    i = 0
    while 'End Field Description' not in data[i]: i += 1

    data = data[i+1:]

    for i in range(1, len(data)):
        info = data[i].split()
        siteid = info[0]
        for_json.setdefault(siteid, {})
        dt = [int(pt) for pt in info[1:6]]
        moment = str(tuple(dt[0:5]))
        for_json[siteid].setdefault("breaks", {})
        for_json[siteid]["breaks"][moment] = {"offsets": [float(pt) for pt in info[6:12]],
                                              "type": info[12],
                                              "description": " ".join(info[13:]),}
    ###############

    # Read Velocities MOD TAH: Replace final with snaps file.
    for velf in ('cwu.snaps_igs14.vel','cwu.fanet_igs14.vel') :
        if ( esver[0] == 0 ):
            get_es_file("https://gage-data.earthscope.org/archive/gnss/products/velocity/"+velf)
        # https://gage-data.earthscope.org/archive/gnss/products/velocity/cwu.fanet_igs14.vel
        else :
            fname = velf
            outpath = DATA_DIR / "NGF" / velf
            url = "https://gage-data.earthscope.org/archive/gnss/products/velocity/"+velf
            print('Calling fetch_csv with ',url,fname)
            url, ok, err = fetch_csv(url, outpath, headers=headers)
            if ok:
                print(f"✓ {fname}")
            else:
                print(f"✗ {fname}  --  {err}")


        with open(DATA_DIR / "NGF" / velf, "r") as f:
            data = list(f)

        i = 0
        while 'End Field Description' not in data[i]: i += 1

        data = data[i+1:]

        for i in range(1, len(data)):
            info = data[i].split()
            siteid = info[0]
            for_json.setdefault(siteid, {})
            for_json[siteid]["velocity"] = [float(pt)*1000 for pt in info[19:22]] # switch to mm
            for_json[siteid]["velsig"] = [float(pt)*1000 for pt in info[22:25]]
            # Get positions from velocity file 8,9,10 lat, long, height
            if( float(info[8])>180 ):
                for_json[siteid]["location"] = [float(info[7]),float(info[8])-360]
            else:
                for_json[siteid]["location"] = [float(info[7]),float(info[8])]

            for_json[siteid]["height"] = float(info[9])


    ###############

    global data_of
    data_of["NGF"] = for_json

    with open(DATA_DIR / 'NGF' / 'NGF_data.json', 'w') as f:
        json.dump(for_json, f)

    with fetcher_output:
        print("Latest NGF Data Downloaded")
update_NGF_butt.on_click(UpdateNGF)


#### Availability

In [ ]:
def site_search(_button):
    searched = site_searchbar.value.strip().upper()
    with availability_output:
        try:
            print(searched + "(" + org_avail_select.value + ")", ": ")
            org = org_avail_select.value
            long_tuple = data_of[org][searched]
            long_string = str(long_tuple)
            wrapped_string = textwrap.wrap(long_string, width=150)
            for line in wrapped_string:
                print(line)
        except:
            display(wg.HTML('''<em style="color:red">Not Found!</em>'''))

site_search_submit.on_click(site_search)


def clear_avail_log(_button):
    availability_output.clear_output()
clear_log_butt.on_click(clear_avail_log)

def add_breaks_data_from_json(_button):
    org = breaks_select.value
    for site in data_of[org]:
        if site in data_of["NGF"]  and "breaks" in data_of["NGF"][site]:
            data_of[org][site]["breaks"] = data_of["NGF"][site]["breaks"]
    with breaks_update_output:
        display(f"Breaks added to {org} data")
    print(f"Breaks added to {org} data")

add_breaks_site_button.on_click(add_breaks_data_from_json)

def add_indv_breaks_data_from_NGF(_button):
    selected = list(ts_sites.value)
    for site in selected:
        siteid, org = remove_brac(site) # remove brac takes "P234 (UNR)" and returns ("P234", "UNR")
        if siteid in data_of["NGF"]:
            data_of[org][siteid]["breaks"] = data_of["NGF"][siteid]["breaks"]
    with breaks_data_output:
        display(f"NGF breaks data added to {siteid} ({org}) data")

add_breaks_ts_button.on_click(add_indv_breaks_data_from_NGF)

def remove_breaks_data_from_json(_button):
    org = breaks_select.value
    for site in data_of[org]:
        try:
            del data_of[org][site]["breaks"]
        except:
            pass
    with breaks_update_output:
        display(f"Breaks removed from {org} data")

remove_breaks_site_button.on_click(remove_breaks_data_from_json)

def remove_indv_breaks_data_from_NGF(_button):
    selected = list(ts_sites.value)
    for site in selected:
        siteid, org = remove_brac(site) # remove brac takes "P234 (UNR)" and returns ("P234", "UNR")
        try:
            del data_of[org][siteid]["breaks"]
        except:
            pass
        display(f"NGF breaks data removed from {siteid} ({org}) data")
remove_breaks_ts_button.on_click(remove_indv_breaks_data_from_NGF)

#### Map

In [ ]:
## TODO: main map workflow ported to ipyleaflet; fixed bottom-left vector guide added
# Map Functions
# Choices of maps are roughly the same as the old Folium version.

vector_guide = HTML()


def update_vector_guide(_change: object = None) -> None:
    """Updates the fixed velocity guide using the current map zoom and scale settings.

    Args:
        _change: optional widget change event.
    """
    # Fixed guide that stays in the lower-left corner of the map.
    if not hasattr(map[0], 'zoom'):
        return

    guide_vel = arrow_length_input.value
    scale = velocity_scale_input.value
    guide_km = guide_vel * scale
    center_lat = map[0].center[0] if map[0].center else 0
    meters_per_pixel = 156543.03392 * np.cos(np.deg2rad(center_lat)) / (2 ** map[0].zoom)
    raw_px = guide_km * 1000 / max(meters_per_pixel, 1e-6)
    bar_px = min(max(raw_px, 20), 220)
    note = '' if raw_px <= 220 else '<br><span style="color:#666">bar shortened to fit panel</span>'

    vector_guide.value = (
        '<div style="background:white; padding:8px; border:1px solid #888; font-size:12px; line-height:1.25">'
        f'<b>{guide_vel:.0f} mm/yr</b> vector guide<br>'
        f'<div style="width:{bar_px:.0f}px; height:5px; background:black; margin:6px 0 3px 0"></div>'
        f'{guide_km:.0f} km at current scale{note}'
        '</div>'
    )


def add_vector_guide(map_obj: Map) -> None:
    """Adds the fixed velocity guide to the lower-left corner of the map.

    Args:
        map_obj: ipyleaflet map object
    """
    # Put the guide on the map as a screen-fixed widget, not a geographic vector.
    update_vector_guide()
    map_obj.add(WidgetControl(widget=vector_guide, position='bottomleft'))
    map_obj.observe(update_vector_guide, names='zoom')
    map_obj.observe(update_vector_guide, names='center')


def BaseMap_Set(_button: object) -> object:
    """Selects the ipyleaflet basemap layer from the basemap widget value.

    Args:
        _button: button click event.

    Returns:
        The selected ipyleaflet basemap or TileLayer.
    """
    if BaseMap_butt.value == 'OpenTopo':
        return TileLayer(
            url="https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png",
            attribution=(
                'Map data: &copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> '
                'contributors, <a href="http://viewfinderpanoramas.org">SRTM</a> | '
                'Map style: &copy; <a href="https://opentopomap.org">OpenTopoMap</a>'
            ),
            name='OpenTopoMap'
        )
    elif BaseMap_butt.value == 'OpenStreetMap':
        return basemaps.OpenStreetMap.Mapnik
    else:
        return basemaps.Esri.WorldImagery


def new_map(b):
    base_map = BaseMap_Set(b)

    ll = (40,-100) ; zoom = 4
    for option in list(map_radius_list.value):
        option = option.split(", ")
        ll = data_of[option[1]][option[0]]["location"]
        zoom = 6

    if hasattr(map[0], 'layers'):
        map_dynamic_layers[0].clear_layers()
        map[0].layout = Layout(width='100%', height='600px', margin='2% 0')
        map[0].basemap = base_map
        map[0].center = ll
        map[0].zoom = zoom
        update_vector_guide()
        return

    map[0] = Map(
        center=ll,
        zoom=zoom,
        basemap=base_map,
        scroll_wheel_zoom=True,
        layout=Layout(width='100%', height='600px', margin='2% 0')
    )
    map_dynamic_layers[0] = LayerGroup(name='Station and Velocity Layers')
    map[0].add(map_dynamic_layers[0])
    map[0].add(WidgetControl(widget=map_loading, position='topright'))
    add_vector_guide(map[0])
    with map_output:
        display(map[0])
new_map_butt.on_click(new_map)

def reload_map(_button):
    """Builds replacement layers off-map, then reveals them together.

    Args:
        _button: reload button click event, or None for a programmatic refresh.
    """
    if not hasattr(map[0], 'layers') or map_reloading[0]:
        return

    map_reloading[0] = True
    reload_map_butt.disabled = True
    map_loading.layout.display = 'block'
    try:
        map[0].basemap = BaseMap_Set(_button)
        stage_layer_update(map_dynamic_layers[0], _render_map_selection)
    finally:
        map_loading.layout.display = 'none'
        reload_map_butt.disabled = False
        map_reloading[0] = False

reload_map_butt.on_click(reload_map)

def map_setting_changed(_change):
    """Refreshes visible map data after a discrete setting changes.

    Args:
        _change: widget change event for a map setting.
    """
    reload_map(None)

BaseMap_butt.observe(map_setting_changed, names='value')

def close_map(_button):
    map_output.clear_output()
    nearest_site_output.clear_output()
    graph_output.clear_output()
    map[0] = "Not initialized yet!"
    map_dynamic_layers[0] = None
close_map_butt.on_click(close_map)


####################
# SITE/RADIUS LIST #
####################

def update_map_list(_button):
    siteid = site_id.value.strip().upper()
    radius = int(site_radius.value or 0)
    org = org_map_select.value
    if siteid in data_of[org].keys():
        addition = f"{siteid}, {org}, {radius}km"
        first_site = not map_radius_list.options
        map_radius_list.options = list(map_radius_list.options) + [addition]
        if first_site:
            map_radius_list.value = (addition,)
        render_site_table()
        render_neighbor_table()
    else:
        print("Not A Valid Site!")

    # MOD TAH 241223: Don't clear site name

add_sites_map_button.on_click(update_map_list)

def remove_map_list(_button):
    options = list(map_radius_list.options)
    for option in map_radius_list.value:
        options.remove(option)
    map_radius_list.options = options
    map_radius_list.value = tuple(option for option in map_radius_list.value if option in options)
    render_site_table()
    render_neighbor_table()

remove_sites_map_button.on_click(remove_map_list)

def sync_map_site_selection(_change):
    """Copies selected table rows into the map's selected-site list.

    Args:
        _change: station-row selection change event.
    """
    map_radius_list.value = tuple(
        option for option, check in map_site_checks.items() if check.value
    )
    render_neighbor_table()

def site_table_cell(widget, width):
    """Places one widget inside a bordered station-table cell.

    Args:
        widget: widget to display in the cell.
        width: CSS width for the cell.

    Returns:
        A boxed widget with the station-table cell styling.
    """
    cell = wg.Box([widget], layout=wg.Layout(
        width=width, border='1px solid #b8b8b8', padding='2px 5px',
        height='26px', min_height='26px', max_height='26px', overflow='hidden',
        justify_content='center', align_items='center'
    ))
    cell.add_class('gnss-table-cell')
    return cell

def table_header(text):
    """Builds one compact, non-wrapping header label for either table.

    Args:
        text: heading text to display.

    Returns:
        An HTML widget containing the formatted heading.
    """
    return wg.HTML(
        f'<b style="font-size: 0.80em; white-space: nowrap;">{text}</b>'
    )

def station_number_toggle(number, selected):
    """Builds a numbered station selector that greys itself when selected.

    Args:
        number: one-based row number to display.
        selected: whether the station row starts selected.

    Returns:
        A toggle button for selecting the station row.
    """
    selector = wg.ToggleButton(
        value=selected, description=str(number),
        layout=wg.Layout(width='100%', height='100%', padding='0')
    )

    def update_selector_style(change):
        selector.style.button_color = '#d9d9d9' if change['new'] else None

    update_selector_style({'new': selected})
    selector.observe(update_selector_style, names='value')
    return selector

def render_site_table():
    """Renders five visible station rows, expanding when additional rows are needed."""
    map_site_checks.clear()
    widths = ['55px', '100px', '85px', '95px']
    title_row = HBox([
    site_table_cell(
        wg.HTML('<div style="text-align:center;"><b>List of Selected Stations</b></div>'),
        '335px'
    )
], layout=wg.Layout(gap='0'))
    header = HBox([
        site_table_cell(table_header('No.'), widths[0]),
        site_table_cell(table_header('Station ID'), widths[1]),
        site_table_cell(table_header('Source'), widths[2]),
        site_table_cell(table_header('Radius (km)'), widths[3]),
    ], layout=wg.Layout(gap='0'))
    rows = [title_row, header]
    row_count = max(5, len(map_radius_list.options))
    for number in range(1, row_count + 1):
        if number <= len(map_radius_list.options):
            option = map_radius_list.options[number - 1]
            siteid, org, radius = option.split(', ')
            selector = station_number_toggle(number, option in map_radius_list.value)
            map_site_checks[option] = selector
            selector.observe(sync_map_site_selection, names='value')
        else:
            selector = wg.Label(str(number))
            siteid, org, radius = '', '', ''
        rows.append(HBox([
            site_table_cell(selector, widths[0]),
            site_table_cell(wg.Label(siteid), widths[1]),
            site_table_cell(wg.Label(org), widths[2]),
            site_table_cell(wg.Label(radius), widths[3]),
        ], layout=wg.Layout(gap='0')))
    site_table.children = tuple(rows)

def nearest_station_rows(org, siteid, count=10):
    """Returns the nearest other stations, excluding the selected station itself.

    Args:
        org: data source name.
        siteid: station code used as the reference location.
        count: maximum number of neighboring stations to return.

    Returns:
        Station records ordered from nearest to farthest.
    """
    candidates, _ = nearby_sites(org, siteid, float('inf'))
    others = [row for row in candidates if row[0] != siteid]
    return sorted(others, key=lambda row: row[3])[:count]

def render_neighbor_table():
    """Renders ten nearest-neighbor rows for the first selected station."""
    widths = ['55px', '100px', '85px', '95px']
    title_row = HBox([
    site_table_cell(
        wg.HTML('<div style="text-align:center;"><b>List of Nearest Neighbors</b></div>'),
        '335px'
    )
], layout=wg.Layout(gap='0'))
    header = HBox([
        site_table_cell(table_header('No.'), widths[0]),
        site_table_cell(table_header('Station ID'), widths[1]),
        site_table_cell(table_header('Source'), widths[2]),
        site_table_cell(table_header('Distance (km)'), widths[3]),
    ], layout=wg.Layout(gap='0'))
    neighbours = []
    if map_radius_list.value:
        siteid, org, _radius = map_radius_list.value[0].split(', ')
        neighbours = nearest_station_rows(org, siteid)
    rows = [title_row, header]
    for number in range(1, 11):
        if number <= len(neighbours):
            siteid, _lat, _lon, distance = neighbours[number - 1]
            org = map_radius_list.value[0].split(', ')[1]
            distance_text = f'{distance:.2f}'
        else:
            siteid, org, distance_text = '', '', ''
        rows.append(HBox([
            site_table_cell(wg.Label(str(number)), widths[0]),
            site_table_cell(wg.Label(siteid), widths[1]),
            site_table_cell(wg.Label(org), widths[2]),
            site_table_cell(wg.Label(distance_text), widths[3]),
        ], layout=wg.Layout(gap='0')))
    neighbor_table.children = tuple(rows)

def parse_bulk_selection(text):
    """Parses individual row numbers and bracketed inclusive or exclusive ranges.

    Args:
        text: comma, space, or line-break delimited row numbers and ranges.
            Parentheses exclude an endpoint; square brackets include it.

    Returns:
        A set of one-based station-row numbers selected by the input.
    """
    return core_parse_bulk_selection(text)

def apply_bulk_selection(_button):
    """Selects listed station rows, then refreshes the sites and neighbors tables.

    Args:
        _button: Select button click event.
    """
    selected_numbers = parse_bulk_selection(bulk_selection_input.value)
    map_radius_list.value = tuple(
        option for number, option in enumerate(map_radius_list.options, start=1)
        if number in selected_numbers
    )
    render_site_table()
    render_neighbor_table()

def select_all_sites(_button):
    """Selects every station currently listed in the sites table.

    Args:
        _button: Select All button click event.
    """
    map_radius_list.value = tuple(map_radius_list.options)
    render_site_table()
    render_neighbor_table()

select_bulk_sites_butt.on_click(apply_bulk_selection)
select_all_sites_butt.on_click(select_all_sites)

################
# SITE BUTTONS #
################

def _render_map_selection(map_obj: object):
    """Draws selected stations once, keeping their markers above vector paths.

    Args:
        map_obj: map layer group that receives the rendered circles, vectors, and markers.
    """
    selected_options = parse_map_selection(map_radius_list.value)
    plotvec = bool(plot_vec_check.value)
    siglim = number_list(vel_siglim_form.value, [1, 1, 2])
    up_limit = colorscalefactor_input.value
    nearby_records = {}
    circle_centers = {}

    for (org, siteid), radius in selected_options.items():
        if radius > 0:
            nearby_records[(org, siteid)], circle_centers[(org, siteid)] = nearby_sites(org, siteid, radius)

    render_plan = build_map_render_plan(
        selected_options, nearby_records, plotvec, thin_velocities_input.value
    )
    for key in render_plan.circle_keys:
        basic_circle(map_obj, circle_centers[key], selected_options[key])
    for org, siteid in render_plan.vector_keys:
        plot_velocity(siteid, org, siglim, map_obj)

    # Add markers after vectors so their popup targets remain on top.
    for org, siteid in render_plan.marker_keys:
        value = get_vel(siteid, org)
        tooltip = f"{siteid}, no vel. data" if not isinstance(value, tuple) else f"{siteid}, vel:{value[0]}"
        selected = (org, siteid) in selected_options
        plot_site_locations(
            map_obj, org, [data_of[org][siteid]["location"]], [siteid], [tooltip],
            color="black" if selected else "blue", dotrad=4 if selected else 3
        )

    update_vector_guide()

def site_circle(b):
    if not hasattr(map[0], 'layers'):
        new_map(b)
    reload_map(b)

site_circle_submit.on_click(site_circle)


def nn_graph(_button):
    nearest_site_output.clear_output()
    graph_output.clear_output()
    selected_options = {}
    for option in list(map_radius_list.value):
        option = option.split(", ")
        selected_options[option[0]] = {"org":option[1], "radius":int(option[2][0:-2])}


    for site in selected_options:
        if selected_options[site]["radius"] > 0:
            siteid = site
            rad = selected_options[site]["radius"]
            org = selected_options[site]["org"]
            site_list, _ = nearby_sites(org, siteid, rad)
            neighbours = sorted(site_list, key = lambda x: x[3])[:10]
            velocities = [] ; relvel = [] ;
            refvel = data_of[org][siteid]["velocity"]

            for site in neighbours:
                if isinstance(get_vel(site[0], org), tuple):
                    vel = get_vel(site[0], org)[0]
                    velocities.append(get_vel(site[0], org))
                    relvel.append(vec_sub(vel,refvel))
                else:
                    velocities.append(("None","None"))
                    relvel.append("None")

            with nearest_site_output:

                display(wg.HTML("""<h2>10 Nearest Neighbouring Sites:</h2>"""))
                tableHTML = f"<table><tr><th><b>Site</b></th><th><b>Distance from {siteid}</b></th><th><b>Velocity (n, e, u)</b></th><th><b>VelSig (n, e, u)</b></th><th><b>RelVel (n, e, u)</b></th></tr>"
                for i in range(len(neighbours)):
                    if ( velocities[i][0] != "None") :
                        velstr = [f"{x:.2f}" for x in velocities[i][0]] ; velflt = [float(x) for x in velstr]
                        sigstr = [f"{x:.2f}" for x in velocities[i][1]] ; sigflt = [float(x) for x in sigstr]
                        dvelstr =  [f"{x:.2f}" for x in relvel[i]] ; dvelflt = [float(x) for x in dvelstr]
                    else:
                        velflt = "None" ; sigflt = "None" ; dvelflt = "None"

                    tableHTML += f"<tr><td>{neighbours[i][0]}</td><td>{neighbours[i][3]:.3f} km</td><td>{velflt}</td><td>{sigflt}</td><td>{dvelflt}</td></tr>"
                display(wg.HTML(tableHTML + "</table>"))

            # See if inline backend (if not create separate figure)
            s = str(plt.get_backend())
            indx = s.find('inline')
            if( indx > 0 ):
                with graph_output:
                    display(wg.HTML("""<h2 style="text-align:center;">Distances to Nearby Sites (km)</h2>"""))
                    plt.bar([x[0] for x in neighbours], [x[3] for x in neighbours])
                    plt.show()
            else:
                print('Creating new figure backend: ',plt.get_backend)
                plt.figure(figsize=(8, 6))
                plt.bar([x[0] for x in neighbours], [x[3] for x in neighbours])
                plt.show()



def get_vel(siteid: str, org: str) -> object:
    """Gets velocity and velocity uncertainty for a station.

    Args:
        siteid: 4-character station code
        org: data source name

    Returns:
        A tuple containing velocity and velocity uncertainty, or None if unavailable.
    """
    # jpl format {"AB01": {"location": [52.209501, -174.204758], "velocity": [-23.472, -7.111, 1.69], "velsig": [0.003, 0.002, 0.009]}
    try:
        return data_of[org][siteid]["velocity"], data_of[org][siteid]["velsig"]
    except:
        return None

def plot_velocity(siteid: str, org: str, siglim: list, map_obj: object = None) -> None:
    """Plots a station velocity vector if its uncertainties are below the limits.

    Args:
        siteid: 4-character station code
        org: data source name
        siglim: maximum allowed North, East, and Up velocity uncertainties
        map_obj: map or layer group where the vector is drawn
    """
    if get_vel(siteid, org): velocity, velsig = get_vel(siteid, org)
    else: return

    # Relative mode subtracts the selected reference station's horizontal velocity.
    if velocity_mode_select.value == 'relative':
        for option in list(map_radius_list.value):
            option = option.split(", ")
            site_ref = option[0]
            refvel = data_of[org][site_ref]["velocity"]

    else:
        refvel = [0,0,0]
    # Only update N and E values (leave height unchanged)
    pltvel = [velocity[0]-refvel[0], velocity[1]-refvel[1], velocity[2]]


    # -10 <=> rgb(0,0,255)
    # 0 <=> rgb(0,0,0)
    # 10 <=> rgb(255,0,0)
    # Get Up velocity scaling factor ±255/colorscalefactor
    # Convert from ±<Range> to colorfactor: Red down; Blue Up
    colorscalefactor = 255/colorscalefactor_input.value

    if all(velsig[i] <= siglim[i] for i in range(3)):
        zv = velocity[2]
        if zv > 0:
            rv, bv = 0, min(255, zv * colorscalefactor)
        else:
            rv, bv = min(255, zv * ( -colorscalefactor)), 0
        rv, bv = int(round(rv)), int(round(bv))
        color = f"#{rv:02x}00{bv:02x}"
        if map_obj is None:
            map_obj = map[0]
        draw_vector(map_obj, data_of[org][siteid]["location"], pltvel, velocity_scale_input.value, color=color)


def draw_length(_button):
    length = arrow_length_input.value
    coords = length_coords_text.value.split(",")
    for i in range(len(coords)):
        coords[i] = float(coords[i])

    end_coords = calc_distance_earth(coords[0], coords[1], length)

    line = Polyline(locations=[coords, end_coords], weight=3)
    line.popup = HTML(value=f"{length} mm/yr")
    map[0].add(line)
    with map_output:
        map_output.clear_output()
        display(map[0])


def clear_map_list(_button):
    map_radius_list.options = []
    map_radius_list.value = ()
    render_site_table()
    render_neighbor_table()
clear_map_list_butt.on_click(clear_map_list)

# Update the fixed vector guide when the scale controls move.
arrow_length_input.observe(update_vector_guide, names='value')
velocity_scale_input.observe(update_vector_guide, names='value')


#### Timeseries

In [ ]:
# Note: This initialization was added to avoid a bug where the backend was not initialized before plotting, which caused errors in some environments.
backend_initialized = False

def update_ts_list(_button):
    siteid = ts_site_form.value.strip().upper()

    if siteid not in data_of[org_ts_select.value]:
        timeseries_output.clear_output()
        with timeseries_output:
            display(wg.HTML('''<em style="color:red">Site not in ''' + org_ts_select.value + '''.</em>'''))
    else:
        extendedsiteid = siteid + " (" + org_ts_select.value + ")"
        if extendedsiteid not in ts_sites.options:
            ts_sites.options = list(ts_sites.options) + [extendedsiteid]


append_butt.on_click(update_ts_list)

def list_to_graph(_button: object) -> None:
    """Plots the currently selected time-series stations from the widget controls.

    Args:
        _button: button click event.
    """
    global tsfig, backend_initialized
    if not backend_initialized:
        backend_act(None)
    selected = list(ts_sites.value)
    timeseries_output.clear_output()

    if len(shift_dict_all) != 0:
        shift_dict_selected = {}
        for site in selected:
            if site in shift_dict_all:
                shift_dict_selected[site] = shift_dict_all[site]
    else:
        shift_dict_selected = {}

    if siglim_form.value:
        global SigLim
        nlim, elim, ulim = number_list(siglim_form.value, [10, 10, 30])
        SigLim = (nlim, elim, ulim)

    yearrange = [datetime(1,1,1), datetime(3000,1,1)] # [far past, far future]

    if selected:
        if start_year_form.value: yearrange[0] = datetime.strptime(start_year_form.value, "%Y-%m-%d")
        if end_year_form.value: yearrange[1] = datetime.strptime(end_year_form.value, "%Y-%m-%d")

        with timeseries_output:
            plot_ts_graph_list(
                selected,
                int(live_update_check.value),
                yearrange,
                breaks = remove_breaks_checkbox.value,
                detrend = detrend_check.value,
                errorbars = [error_bar_check.value, thin_error_bars.value],
                errorbar_outline = [error_bar_outline_check.value, error_bar_opacity.value],
                outlier = remove_outliers.value,
                resolution = plot_ts_res.value,
                shift = shift_dict_selected,
                )
    else:
        with timeseries_output:
            display(wg.HTML('''<em style="color:red">Select a set of sites to plot!</em>'''))


plot_ts_butt.on_click(list_to_graph)

def remove_site(_button):
    options = list(ts_sites.options)
    for site in ts_sites.value:
        options.remove(site)
        shift_dict_all.pop(site, None)
    ts_sites.options = options

remove_site_button.on_click(remove_site)

shift_dict_all = {}

def add_shift(_button):
    selected = list(ts_sites.value)
    shift_output.clear_output()
    if shift_value.value:
        for site in selected:
            shift_dict_all[site] = shift_value.value
    else:
        for site in selected:
            shift_dict_all.pop(site, None)

    with shift_output:
            print("shifts:")
            print(shift_dict_all)

update_customization.on_click(add_shift)

def display_break_data(_button):
    breaks_data_output.clear_output()

    for siteid in ts_sites.value:
        site, org = remove_brac(siteid)
        if "breaks" in data_of[org][site]:
            with breaks_data_output:
                display(wg.HTML(f"{siteid} Break Data"))
                tableHTML = f"<table><tr><th><b>Date (yr, m, d, hr, min)</b></th><th><b>Offset in mm (dN, sN, dE, sE, dU, sU)</b></th><th><b>Type</b></th><th><b>Description</b></th></tr>"
                for i in data_of[org][site]["breaks"]:
                    tableHTML += f'<tr><td>{i}</td><td>{data_of[org][site]["breaks"][i]["offsets"]}</td><td>{data_of[org][site]["breaks"][i]["type"]}</td><td>{data_of[org][site]["breaks"][i]["description"]}</td></tr>'
                display(wg.HTML(tableHTML + "</table>"))
        else:
            with breaks_data_output:
                display(f"{siteid}: No Breaks Data")

show_breaks_data_button.on_click(display_break_data)

def backend_act(_button: object) -> None:
    """Activates the selected matplotlib backend for time-series plotting.

    Args:
        _button: button click event.
    """
    global backend_initialized
    # Note: This seems to be needed to get interactive graphic to appear
    systype = sys.platform
    if( backend_butt.value == 'Inline') :
        try:
            plt.switch_backend('module://matplotlib_inline.backend_inline')
            get_ipython().run_line_magic('matplotlib', 'inline')
        except:
            with timeseries_output:
                print('Problem selecting inline')
    elif ( backend_butt.value == 'ipympl') :
        try:
            plt.close('all')
            get_ipython().run_line_magic('matplotlib', 'widget')
        except Exception as e:
            with timeseries_output:
                print('ipympl error:', e)
    else :
        print('Checking System',systype)
        if (systype=='darwin' ) :
            try:
                plt.switch_backend('macosx')
            except:
                with timeseries_output:
                    print('Problem with osx with ',systype)
        elif (systype == 'posix' or systype == 'linux'):
            try:
                plt.switch_backend('qt5agg')
            except:
                with timeseries_output:
                    print('Problem with qt5agg with ',systype)
        else :
            try:
                plt.switch_backend('qt')
            except:
                with timeseries_output:
                    print('Problem with qt with ',systype)

    backend_initialized = True
    with timeseries_output:
        print('Using backend ',mpl.get_backend(),systype)

backend_activate.on_click(backend_act)

def close_ts(_button):
    timeseries_output.clear_output()
close_ts_butt.on_click(close_ts)

def clear_list(_button):
    ts_sites.options = []
    shift_dict_all.clear()
clear_list_butt.on_click(clear_list)

def id_points(_button: object) -> None:
    """Lets the user click plotted points and prints offset rename commands.

    Args:
        _button: button click event.
    """
    global tsfig
    username = os.environ.get('USER', os.environ.get('USERNAME'))
    if ( backend_butt.value == 'Interactive') :
        pts = tsfig.ginput(n=10, show_clicks=True)
        cnt = 0
        for pt in pts :
            cnt += 1
            dt = pt[0]*86400
            epoch = datetime.fromtimestamp(dt,tz=timezone.utc).strftime("%Y %m %d %H %M")
            ptstr = f"Point {cnt:2d} Epoch {epoch} Value {pt[1]:.2f} mm"
            with timeseries_output:
                print(ptstr)

        cnt = 0
        for pt in pts :
            dt = pt[0]*86400
            epoch = datetime.fromtimestamp(dt,tz=timezone.utc).strftime("%Y %m %d %H %M")
            ax= tsfig.axes
            ax[2].axvline(x=pt[0], color='g', linestyle='dotted', linewidth=2)
            ax[1].axvline(x=pt[0], color='g', linestyle='dotted', linewidth=2)
            ax[0].axvline(x=pt[0], color='g', linestyle='dotted', linewidth=2)
            with timeseries_output:
                siteid = ts_sites.value[0][0:4]
                code = "_"+chr(ord('A') + cnt)+"PS"
                current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                print(" rename", siteid ,"   ",siteid+code,epoch,"                  ! GNSS_Analysis", username,current_time)
            cnt += 1
    else:
        with timeseries_output:
            display(wg.HTML('''<em style="color:red">ID only works with Interactive Backend activated'!</em>'''))

id_button.on_click(id_points)


### Windows Layout

In [ ]:
fetch_window = VBox([
    wg.HTML(
            '<em style="color:blue; line-height: 1.5">Use the buttons below to update the local files with the latest data. This may take a few seconds.</em>'
            ),
    HBox([update_NGF_butt, update_UNR_butt, update_JPL_butt]),
    fetcher_output,
    HBox([breaks_select, add_breaks_site_button, remove_breaks_site_button]),
    HBox([breaks_update_output])
])

availability_window = VBox([
    wg.HTML('<h1>Availability</h1>'),
    wg.HTML(
            '<em style="color:blue; line-height: 1.5"><ul>' +
            '<li>Type the Site ID into the box below to check its existence and status.</li>' +
            '</ul></em>'
            ),
    HBox([site_searchbar, org_avail_select, site_search_submit, clear_log_butt]),
    availability_output
    ])

render_site_table()
render_neighbor_table()
left_content_width = '100%'
left_label_layout = wg.Layout(width='38%')
left_input_row_layout = wg.Layout(
    width=left_content_width, justify_content='space-between', align_items='center'
)
station_input_section = VBox([
    HBox([wg.Label('Station ID:', layout=left_label_layout), site_id], layout=left_input_row_layout),
    HBox([wg.Label('Radius (km):', layout=left_label_layout), site_radius], layout=left_input_row_layout),
    HBox([wg.Label('Source:', layout=left_label_layout), org_map_select], layout=left_input_row_layout),
], layout=wg.Layout(gap='3%'))

compact_action_layout = wg.Layout(width='145px', height='36px')
for action_button in [new_map_butt, reload_map_butt, close_map_butt]:
    action_button.layout = compact_action_layout

# Site-list actions stay with the station inputs they affect.
site_action_layout = wg.Layout(width='48.5%')
add_sites_map_button.layout = site_action_layout
remove_sites_map_button.layout = site_action_layout
clear_map_list_butt.layout = wg.Layout(width='100%')
left_action_grid = VBox([
    HBox([add_sites_map_button, remove_sites_map_button], layout=wg.Layout(
        width='100%', justify_content='space-between'
    )),
    clear_map_list_butt,
], layout=wg.Layout(width='100%', gap='3%'))
base_map_row = HBox([
    wg.Label('Base Map Choice:', layout=left_label_layout), BaseMap_butt,
], layout=left_input_row_layout)
site_circle_submit.layout = wg.Layout(width='100%')
def build_basemap_preview():
    """Creates a non-interactive preview using the current basemap choice.

    Returns:
        An ipyleaflet map widget with interaction controls disabled.
    """
    return Map(
        center=(40.0, -110.0), zoom=4, basemap=BaseMap_Set(None),
        zoom_control=False, attribution_control=False, dragging=False,
        double_click_zoom=False, scroll_wheel_zoom=False, touch_zoom=False,
        box_zoom=False, keyboard=False,
        layout=wg.Layout(width='100%', height='100%')
    )

base_map_preview = build_basemap_preview()
base_map_preview_container = wg.Box([base_map_preview], layout=wg.Layout(
    width='100%', flex='1 1 0', min_height='0', overflow='hidden'
))

def update_basemap_preview(_change):
    """Rebuilds the preview so every basemap source refreshes visibly.

    Args:
        _change: base-map selection change event.
    """
    global base_map_preview
    base_map_preview = build_basemap_preview()
    base_map_preview_container.children = (base_map_preview,)

BaseMap_butt.observe(update_basemap_preview, names='value')
base_map_section = VBox([
    base_map_row, site_circle_submit, base_map_preview_container,
], layout=wg.Layout(
    width='100%', flex='1 1 0', min_height='0', box_sizing='border-box',
    border='1px solid #d0d0d0', padding='4%', gap='4%'
))

vector_label_layout = wg.Layout(width='43%')
vector_field_layout = wg.Layout(width='55%')
vector_slider_layout = wg.Layout(width='34%')
vector_value_layout = wg.Layout(width='18%')
vector_row_layout = wg.Layout(width='100%', justify_content='space-between', align_items='center')

velocity_mode_select.layout = vector_field_layout
vel_siglim_form.description = ''
vel_siglim_form.layout = vector_field_layout
for vector_slider in [thin_velocities_input, arrow_length_input, velocity_scale_input, colorscalefactor_input]:
    vector_slider.description = ''
    vector_slider.readout = False
    vector_slider.layout = vector_slider_layout

decimate_value_input = wg.BoundedIntText(
    value=thin_velocities_input.value, min=thin_velocities_input.min, max=thin_velocities_input.max,
    step=thin_velocities_input.step, layout=vector_value_layout
)
arrow_length_value_input = wg.BoundedFloatText(
    value=arrow_length_input.value, min=arrow_length_input.min, max=arrow_length_input.max,
    step=arrow_length_input.step, layout=vector_value_layout
)
velocity_scale_value_input = wg.BoundedFloatText(
    value=velocity_scale_input.value, min=velocity_scale_input.min, max=velocity_scale_input.max,
    step=velocity_scale_input.step, layout=vector_value_layout
)
up_rate_value_input = wg.BoundedFloatText(
    value=colorscalefactor_input.value, min=colorscalefactor_input.min, max=colorscalefactor_input.max,
    step=colorscalefactor_input.step, layout=vector_value_layout
)
wg.jslink((thin_velocities_input, 'value'), (decimate_value_input, 'value'))
wg.jslink((arrow_length_input, 'value'), (arrow_length_value_input, 'value'))
wg.jslink((velocity_scale_input, 'value'), (velocity_scale_value_input, 'value'))
wg.jslink((colorscalefactor_input, 'value'), (up_rate_value_input, 'value'))

vector_type_row = HBox([
    wg.Label('Vector Type:', layout=vector_label_layout), velocity_mode_select,
], layout=vector_row_layout)
vector_limit_row = HBox([
    wg.Label('S.D. Limit (σ):', layout=vector_label_layout), vel_siglim_form,
], layout=vector_row_layout)
decimate_row = HBox([
    wg.Label('Decimate:', layout=vector_label_layout), thin_velocities_input, decimate_value_input,
], layout=vector_row_layout)
arrow_length_row = HBox([
    wg.Label('Arrow (mm/yr):', layout=vector_label_layout), arrow_length_input, arrow_length_value_input,
], layout=vector_row_layout)
velocity_scale_row = HBox([
    wg.Label('Scale (km/mm/yr):', layout=vector_label_layout), velocity_scale_input, velocity_scale_value_input,
], layout=vector_row_layout)
up_rate_row = HBox([
    wg.Label('±U Rate (mm/yr):', layout=vector_label_layout), colorscalefactor_input, up_rate_value_input,
], layout=vector_row_layout)

site_configuration_section = VBox([
    station_input_section, left_action_grid,
], layout=wg.Layout(
    width='100%', flex='0 0 auto', box_sizing='border-box',
    border='1px solid #d0d0d0', padding='4%', gap='4%'
))

left_map_column = VBox(
    [site_configuration_section, base_map_section],
    layout=wg.Layout(
    width='33.333%', flex='0 0 33.333%', min_width='0',
    height='100%',
    justify_content='flex-start',
    align_items='stretch',
    gap='3%')
)

table_content_width = '335px'  # Sum of the four fixed table-column widths.
bulk_selection_row = HBox([
    bulk_selection_input, select_bulk_sites_butt,
], layout=wg.Layout(width=table_content_width, justify_content='space-between'))
bulk_selection_section = VBox([
    bulk_selection_row, select_all_sites_butt,
], layout=wg.Layout(width=table_content_width, gap='0.5%'))
center_map_column = VBox(
    [site_table, bulk_selection_section, neighbor_table],
    layout=wg.Layout(
        width='33.333%', flex='0 0 33.333%', min_width='0', height='100%',
        box_sizing='border-box', border='1px solid #d0d0d0',
        justify_content='space-between',
        align_items ='center')
)

plot_vec_check.layout = wg.Layout(width='100%')

vector_preview = HTML(layout=wg.Layout(width='100%', flex='1 1 0', min_height='0'))

def update_vector_preview(_change=None):
    """Shows the selected vector style without drawing a map layer.

    Args:
        _change: optional vector-control change event.
    """
    siglim = number_list(vel_siglim_form.value, [1, 1, 2])
    up_limit = colorscalefactor_input.value
    tick_markup = ''
    for tick_x, tick_value in zip(np.linspace(60, 306, 11), np.linspace(-up_limit, up_limit, 11)):
        tick_label = '0' if abs(tick_value) < 1e-9 else f'{tick_value:+g}'
        tick_markup += (
            f'<line x1="{tick_x:g}" y1="234" x2="{tick_x:g}" y2="256" stroke="#333" stroke-width="1"/>'
            f'<text x="{tick_x:g}" y="230" text-anchor="middle" fill="#333" font-size="8">{tick_label}</text>'
        )
    vector_preview.value = (
        '<div style="width:100%; height:100%; display:flex; flex-direction:column; '
        'border:1px solid #d0d0d0; padding:3%; box-sizing:border-box; text-align:center; background:#fafafa">'
        '<svg viewBox="0 0 320 270" width="100%" style="flex:1 1 auto; min-height:0" preserveAspectRatio="xMidYMid meet" aria-label="East north up velocity-vector reference">'
        '<defs><linearGradient id="up-rate-gradient" x1="0%" x2="100%">'
        '<stop offset="0%" stop-color="#ff0000"/><stop offset="50%" stop-color="#000000"/>'
        '<stop offset="100%" stop-color="#0000ff"/></linearGradient></defs>'
        '<line x1="70" y1="160" x2="240" y2="160" stroke="#707070" stroke-width="3" stroke-dasharray="8 7"/>'
        '<line x1="240" y1="160" x2="240" y2="35" stroke="#707070" stroke-width="3" stroke-dasharray="8 7"/>'
        '<circle cx="70" cy="160" r="15" fill="#1677ff" stroke="#000000" stroke-width="2"/>'
        '<line x1="70" y1="160" x2="240" y2="35" stroke="#000000" stroke-width="5" stroke-linecap="round"/>'
        '<polyline points="211,43 240,35 223,66" fill="none" stroke="#000000" stroke-width="5" stroke-linecap="round" stroke-linejoin="round"/>'
        '<text x="248" y="98" dominant-baseline="middle" fill="#000000" font-size="11" font-weight="600">Vₙ = y ± σₙ</text>'
        '<text x="155" y="190" text-anchor="middle" fill="#000000" font-size="13" font-weight="600">Vₑ = x ± σₑ</text>'
        '<circle cx="24" cy="244" r="10" fill="none" stroke="#000000" stroke-width="2"/>'
        '<circle cx="24" cy="244" r="3" fill="#000000"/>'
        '<text x="24" y="266" text-anchor="middle" textLength="48" lengthAdjust="spacingAndGlyphs" fill="#000000" font-size="10" font-weight="600">Vᵤ = z ± σᵤ</text>'
        + tick_markup +
        '<rect x="60" y="239" width="246" height="13" fill="url(#up-rate-gradient)" stroke="#777" stroke-width="1"/>'
        '<text x="183" y="268" text-anchor="middle" fill="#333" font-size="9" font-weight="600">Vᵤ reference (mm/yr)</text>'
        '</svg>'
        '</div>'
    )

update_vector_preview()
velocity_mode_select.observe(update_vector_preview, names='value')
colorscalefactor_input.observe(update_vector_preview, names='value')
vel_siglim_form.observe(update_vector_preview, names='value')

vector_preview_section = VBox([
    vector_type_row, vector_limit_row, decimate_row, arrow_length_row,
    velocity_scale_row, up_rate_row,
    vector_preview,
], layout=wg.Layout(
    width='100%', flex='1 1 0', min_height='0', gap='3%',
    justify_content='flex-start', align_items='center'
))

vector_settings_section = VBox([
    vector_preview_section,
    HBox([plot_vec_check],
        layout=wg.Layout(
        width='100%',
        justify_content='center',
        gap='8px',
    )),
], layout=wg.Layout(
    width='33.333%', flex='0 0 33.333%', min_width='0', height='100%',
    min_height='0', box_sizing='border-box',
    border='1px solid #c8c8c8', padding='2%', gap='3%',
    justify_content='flex-start', align_items='center'
))

station_selection_section = HBox(
    [left_map_column, center_map_column, vector_settings_section],
    layout=wg.Layout(
        width='100%',              # Fills the entire browser width
        height='600px',
        align_items = "stretch",           # Columns share the dashboard height
        justify_content='flex-start', # Three equal-width dashboard columns
        margin='0 0 24px 0'        # Adds 24px of buffer space below this entire row
    ))

# Global map actions apply to the whole canvas, not to one column.
global_map_actions = HBox(
    [new_map_butt, reload_map_butt, close_map_butt],
    layout=wg.Layout(justify_content='center')
)

table_cell_css = wg.HTML('''
<style>
.gnss-table-cell { font-size: 0.85em !important; }
.gnss-table-cell .widget-label,
.gnss-table-cell .widget-html-content,
.gnss-table-cell .widget-toggle-button { font-size: inherit !important; }
</style>
''')

map_window = VBox([
    table_cell_css,
    wg.HTML('<h1 style="text-align:center; margin:0 0 20px 0;">Map</h1>'),
    station_selection_section,
    global_map_actions,
    map_output,
    ])

timeseries_window = VBox([
    wg.HTML('<h1>Timeseries</h1>'),
    HBox([ts_site_form, org_ts_select, append_butt, live_update_check]),
    HBox([ts_sites, VBox
          ([HBox([plot_ts_butt, show_breaks_data_button]),
            HBox([remove_site_button, clear_list_butt]),
            HBox([close_ts_butt,id_button])])
        ]),
    HBox([start_year_form, end_year_form, siglim_form, remove_outliers]),
    HBox([remove_breaks_checkbox, detrend_check]),
    HBox([error_bar_check, error_bar_outline_check]),
    HBox([error_bar_opacity, thin_error_bars, add_breaks_ts_button, remove_breaks_ts_button]),
    HBox([shift_value, update_customization]),
    HBox([shift_output]),
    HBox([backend_butt,backend_activate])
])
timeseries_output_window = VBox([breaks_data_output, timeseries_output])

Tested on the following versions (as of 07/17/2026):

- Python: 3.11.15
- ipyleaflet: 0.20.0
- ipympl: 0.9.8
- earthscope_sdk: 1.5.0
- pandas: 3.0.2
- numpy: 2.4.6
- ipywidgets: 8.1.7
- matplotlib: 3.10.9
- geopy: 2.4.1
- requests: 2.34.2


## Interface and Downloads

To use the program for the first time, you must run the fetch_window display and click the "Latest" button for every source you want data for. This will download the data of all sites for the source (excluding time-series data) from the relevant website into the "data" folder. Upon using this program again, you may skip this step if you don't want to update the data from the web. The data, once downloaded, will show up in the data/"source" folder as "site"_data.json. 

## Availability

The "Check Availability" box allows you to if data exists for a site in a given source. It will directly output the data for that site as it is stored in the JSON file (in the form of a dictionary). 

Adding/Remove Breaks: If for all sites in the selected organization, if the site is also available in the NGF data, it will take the NGF break data and add it to the non-NGF version of the site, allowing for the "remove breaks" and "show breaks data" button to be available for the graphing portion of the notebook. A way to do this for individual sites can be found in the time-series section below. 



In [ ]:
display(fetch_window, availability_window) # run to display fetcher and availability options

## Using The Map

Adding A Station: Inputting the combination of a valid site id, a desired marker observation radius, selecting a source of data origin, and then clicking "Add Site" will add the given station to the stations table. 

Stations Table: Here, select all stations you wish to interact with: To map the selected sites, click the "Plot Site(s)" button. This will show the sites as well as all other sites within the selected radii. To remove the selected sites from the table, click the "Remove Site(s)" button. To clear all sites from the stations table, click the "Clear All" button.

Distance Table: This table shows the distances of the 10 nearest neighboring sites to the center site. 
It also displays their velocities, velocity relative to the first site, and the velocity sigmas.  

Decimate: Thins out velocity vectors on the map. The input indicates how many "1 in x" vectors will be shown (i.e. if it's 10, then 1 in every 10 vectors will be displayed on the map).

Plot Velocities: Plots vectors showing station velocities. Absolute vectors show each station's individual velocity, while relative vectors show velocities relative to selected site(s).

Arrow: Sets the length of the velocity vector arrow.

Scale: Sets the scale of velocities projection on the map. Nominally, it relates mm/yr to kilometers on the map (i.e. inputting 10 sets 1 mm/yr to be the equivalent of 10 km on the map).  

The color of the vectors shows vertical motions with <span style="color:red">reds indicating downward motion</span>, <span style="color:blue">blues indicating upwards motion</span>, and <span style="color:grey">blacks indicating near zero vertical motion</span>.  

±U Rate: Sets the range at which the "up velocity" color saturates

Base Map Choice: sets the base map with topography, street map, or ArcGIS imagery.



## Using the Timeseries
Site Id + Source Dropdown + Add to list: Input a Site ID with the source you want to add to the Site List. 

Live Update: Check to download the latest time-series data for selected sites, even if a previous version of the data already exists in the Data/ folder. 

Site List + Plot!: Select the Sites to be graphed in the Site List Box, and click "Plot!" to graph them. Selecting multiple sites will overlay their data on top of each other. 

Show Breaks Data: This button will show table(s) with all the breaks data available for the site(s), such as time, cause, type, and offsets. 

Remove Site: Removes the selected sites from the graph. 

Clear List + Close Graph: Self-explanatory.

Start/End: Set start and end dates for the data in the graph. Will only accept dates in the yyyy-mm-dd or yyyy-m-d format. 

SigLim: The limits on the standard deviations of the points of the time-series plot expressed in (north, east, up). Sigma must be less than the values of each component. 

Remove N σ: Only works if Detrend is also checked. Removes outlier data from the graphs. The number determines how strict the outlier determination is; all data outside the standard deviation multiplied by the Remove N σ number will be removed. The data is then detrended and run through the process again until the beginning and ending number of data points remain the same. The number of removed data points and the number of iterations will be given. 

Error Bars + Error Bar Outlines: Shows Error Bars and Error Bar outlines for the data. 

Error Bar Opacity + NErrBar: Changes the opacity of error bars and shows only every N'th errorbar. 

Shift Values + Update Shift: Shifts all the values of the selected sites up or down by the given mm. It is useful to align or separate two or more graph lines for better comparison. Click "Update Shift" to add the given shift value to the site(s). A dictionary displaying the offsets for each site is shown in the window  (if it is not shifted, it will not show up in the dictionary).  All sites can be selected to set the shift back to zero.

Copy/Clear Breaks Data: If the selected sites are also available in the NGF data, it will take the NGF break data and add it to the non-NGF version of the site, allowing for the "remove breaks" and "show breaks data" button to be available. Copying the NGF breaks can eb done for all sites in JPL/UNR time-series can be found in the availability section above. 

Backend Choice allows of type of time-series graphics.  Backend Activate <b>must</b> be used to activate the chosen backend.  
Errors activating the chosen BackEnd are displayed at the bottom of the time-series output window.  The ipympl Backend may need additional package installation.  This Backend allows the time-series plot to be Zoomed and Saved.  The Interactive Backend allows Zooming, Saving, and coordinates to be identified on the time-series plot.  When interactive is first selected, make sure to activate the Backend and check the error to see that it is activated without errors. 

On some systems, once the Backend is changed, the kernel needs to be restarted to change to a different backend.

ID points: Allows locations on the time-series plot to be identified only when the interactive backend is used.  The output includes the coordinates of the locations selected, and GLOBK site rename commands that can be used to add times of breaks in GLOBK/TSFIT solutions.  Multiple points can be selected with the middle mouse button/return ending selections, the right mouse button/delete removing points, and the left button/click selecting next point.  The graphics window needs to be active before the first point is selected. 

Detrended output example

| Site      |    Vn |   σVn |     Ve |   σVe |    Vu |   σVu |   WRMS N |   χn |   WRMS E |   χe |   WRMS U |   χu |   Num |
|:----------|------:|------:|-------:|------:|------:|------:|---------:|-----:|---------:|-----:|---------:|-----:|------:|
| P040-UNR  | -4.84 | 0.002 | -14.39 | 0.001 | -0.07 | 0.006 | 1.55     | 1.87 | 1.61     | 2.38 | 5.57     | 2.08 | 7146  |
| P040-JPL  | -4.86 | 0.002 | -14.41 | 0.001 | -0.01 | 0.005 | 1.66     | 2.24 | 1.54     | 2.54 | 5.41     | 2.26 | 7031  |
| P040-NGF | -4.96 | 0.004 | -14.34 | 0.003 | -0.32|  0.015 | 0.86     | 0.44 | 0.85     | 0.53 | 6.17     | 0.86 | 7262  |



In [ ]:
display(map_window) # displays map and map settings

In [ ]:
display(timeseries_window) # displays settings for graph

In [ ]:
display(timeseries_output_window) # displays graph, breaks data, and shift tracker (tied to "Update Shift")